# Fase 1: Preprocesamiento FF++ (protocolo oficial)

Etapa 1 del pipeline para todos los videos: muestreo uniforme N=16 → landmarks MediaPipe → alineación al eje
central → rostro 224×224, más el índice de giro de la cabeza por frame. Partición oficial de FaceForensics++
(720/140/140, sin fugas de identidad); la normalización robusta se ajusta solo con train.

- Train: 16 frames por real y 4 por fake (equilibra las clases 1:4 de FF++); val: 8; test: 16 (decisión por video con N=16).
- Si el dataset incluye la versión C40, se extraen también sus videos de test (mala calidad real).
- Si se añade Celeb-DF v2, se extrae su lista oficial de test (518 videos) para evaluar generalización.

Esta fase no usa GPU. Sus salidas (cachés + `manifest.json`) son la entrada de la fase 2.

> Modo smoke: 10 videos por conjunto y 1 época; solo verifica que el flujo funciona. Sus métricas no son resultados.

In [ ]:
import os, sys, json, time, glob, subprocess, platform, warnings, random
warnings.filterwarnings("ignore", category=UserWarning)
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("GLOG_minloglevel", "3")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
ON_KAGGLE = os.path.exists("/kaggle/input")
SMOKE = True
WORK_DIR = "/kaggle/working" if ON_KAGGLE else os.path.abspath(os.environ.get("DF_WORK_DIR", "salida_fases"))
os.makedirs(WORK_DIR, exist_ok=True); os.chdir(WORK_DIR); os.makedirs("code_kaggle", exist_ok=True)
INPUT_ROOT = "/kaggle/input" if ON_KAGGLE else os.environ.get("DF_INPUT_ROOT", ".")

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
HAS_NVIDIA = subprocess.run("nvidia-smi", shell=True, capture_output=True).returncode == 0
if ON_KAGGLE and True:
    try:
        import mediapipe  # noqa
    except ImportError:
        _pip("mediapipe")
if ON_KAGGLE and False and HAS_NVIDIA:
    try:
        import cupy  # noqa
    except ImportError:
        _pip("cupy-cuda12x")
if not False:
    os.environ["DF_BACKEND"] = "numpy"      # fases sin entrenamiento: CPU
import numpy as np
random.seed(SEED); np.random.seed(SEED)
print(f"Entorno: {'Kaggle' if ON_KAGGLE else 'local'} | GPU NVIDIA: {HAS_NVIDIA} | smoke: {SMOKE} | Python {platform.python_version()}")

In [ ]:
%%writefile code_kaggle/backend.py
"""Backend numérico: CuPy en GPU o NumPy en CPU (DF_BACKEND=numpy fuerza CPU)."""
from __future__ import annotations

import os
import random
from typing import Any

import numpy as np

_cp = None
if os.environ.get("DF_BACKEND", "").lower() != "numpy":
    try:
        import cupy as _cp  # type: ignore

        if _cp.cuda.runtime.getDeviceCount() < 1:
            _cp = None
    except Exception:  # CuPy no instalado o sin GPU visible
        _cp = None

xp: Any = _cp if _cp is not None else np
ON_GPU: bool = _cp is not None

# float32 para entrenar; float64 solo para la verificación numérica de gradientes.
DTYPE: Any = np.float32
SEED: int = 42


def set_dtype(dtype: Any) -> None:
    """Cambia la precisión global."""
    global DTYPE
    DTYPE = dtype


def set_seed(seed: int = SEED) -> None:
    """Fija todas las semillas (python, hash, numpy y cupy)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    if ON_GPU:
        xp.random.seed(seed)


def asarray(a: Any) -> Any:
    """Convierte a arreglo del backend activo con la precisión global."""
    return xp.asarray(a, dtype=DTYPE)


def to_numpy(a: Any) -> np.ndarray:
    """Devuelve siempre un np.ndarray en CPU (para métricas con sklearn / gráficas)."""
    return _cp.asnumpy(a) if ON_GPU else np.asarray(a)


def randn(*shape: int) -> Any:
    return xp.random.standard_normal(shape).astype(DTYPE)


def rand(*shape: int) -> Any:
    return xp.random.random_sample(shape).astype(DTYPE)


In [ ]:
%%writefile code_kaggle/layers.py
"""Capas básicas con forward y backward implementados a mano."""
from __future__ import annotations

import math
from typing import Any, Iterable, List, Optional, Sequence, Tuple

import numpy as np

import backend as bk
from backend import xp

Array = Any


class Parameter:
    """Tensor entrenable con su gradiente."""

    def __init__(self, data: Array, decay: bool = True) -> None:
        self.data: Array = data.astype(bk.DTYPE)
        self.grad: Array = xp.zeros_like(self.data)
        self.decay: bool = decay
        self.lr_mult: float = 1.0  # multiplicador de lr (p. ej. backbone preentrenado)


class Module:
    training: bool = True

    def forward(self, *args: Array) -> Array:  # pragma: no cover
        raise NotImplementedError

    def backward(self, dy: Array) -> Array:  # pragma: no cover
        raise NotImplementedError

    def __call__(self, *args: Array) -> Array:
        return self.forward(*args)

    def children(self) -> Iterable["Module"]:
        for v in vars(self).values():
            if isinstance(v, Module):
                yield v
            elif isinstance(v, (list, tuple)):
                for m in v:
                    if isinstance(m, Module):
                        yield m

    def parameters(self) -> List[Parameter]:
        ps = [v for v in vars(self).values() if isinstance(v, Parameter)]
        for c in self.children():
            ps += c.parameters()
        return ps

    def train(self, mode: bool = True) -> "Module":
        self.training = mode
        for c in self.children():
            c.train(mode)
        return self

    def eval(self) -> "Module":
        return self.train(False)

    def zero_grad(self) -> None:
        for p in self.parameters():
            p.grad[...] = 0

    def n_params(self) -> int:
        return int(sum(p.data.size for p in self.parameters()))

    # Arreglos que son estado del modelo (no activaciones) y no deben liberarse.
    _PERSISTENT = frozenset({"run_mean", "run_var", "_idx", "_P", "mask"})

    def release(self) -> None:
        """Libera las activaciones guardadas para el backward (de este módulo y sus hijos)."""
        for k, v in list(vars(self).items()):
            if k not in self._PERSISTENT and isinstance(v, (np.ndarray, xp.ndarray)):
                delattr(self, k)
        for c in self.children():
            c.release()


class Sequential(Module):
    def __init__(self, *mods: Module) -> None:
        self.mods: List[Module] = list(mods)

    def forward(self, x: Array) -> Array:
        for m in self.mods:
            x = m(x)
        return x

    def backward(self, dy: Array) -> Array:
        for m in reversed(self.mods):
            dy = m.backward(dy)
        return dy


# Capas densas

class Linear(Module):
    """y = x W + b, con x de forma (..., in) y W de forma (in, out)."""

    def __init__(self, in_f: int, out_f: int, bias: bool = True, init: str = "torch") -> None:
        if init == "trunc":
            w = xp.clip(bk.randn(in_f, out_f) * 0.02, -0.04, 0.04)
            b = xp.zeros(out_f)
        else:
            k = 1.0 / math.sqrt(in_f)
            w = (bk.rand(in_f, out_f) * 2 - 1) * k
            b = (bk.rand(out_f) * 2 - 1) * k
        self.W = Parameter(w)
        self.b: Optional[Parameter] = Parameter(b, decay=False) if bias else None
        self.in_f, self.out_f = in_f, out_f

    def forward(self, x: Array) -> Array:
        assert x.shape[-1] == self.in_f, f"Linear esperaba (..., {self.in_f}), llegó {x.shape}"
        self.x = x
        # Un solo GEMM 2D (mucho más rápido que el matmul por lotes en NumPy/CuPy)
        y = (x.reshape(-1, self.in_f) @ self.W.data).reshape(x.shape[:-1] + (self.out_f,))
        return y + self.b.data if self.b is not None else y

    def backward(self, dy: Array) -> Array:
        x2 = self.x.reshape(-1, self.in_f)
        d2 = dy.reshape(-1, self.out_f)
        self.W.grad = x2.T @ d2
        if self.b is not None:
            self.b.grad = d2.sum(axis=0)
        return (d2 @ self.W.data.T).reshape(self.x.shape)


class ReLU(Module):
    def forward(self, x: Array) -> Array:
        self.mask = x > 0
        return x * self.mask

    def backward(self, dy: Array) -> Array:
        return dy * self.mask


class GELU(Module):
    """GELU (aprox."""

    _C = math.sqrt(2.0 / math.pi)

    def forward(self, x: Array) -> Array:
        self.x = x
        x2 = x * x
        self.t = xp.tanh(self._C * x * (1.0 + 0.044715 * x2))
        return 0.5 * x * (1.0 + self.t)

    def backward(self, dy: Array) -> Array:
        x, t = self.x, self.t
        du = self._C * (1.0 + 3 * 0.044715 * (x * x))
        return dy * (0.5 * (1.0 + t) + 0.5 * x * (1.0 - t * t) * du)


class Dropout(Module):
    """Dropout invertido: en train escala por 1/(1-p); en eval es la identidad."""

    def __init__(self, p: float = 0.3) -> None:
        assert 0.0 <= p < 1.0
        self.p = p
        self.mask: Optional[Array] = None

    def forward(self, x: Array) -> Array:
        if not self.training or self.p == 0.0:
            self.mask = None
            return x
        self.mask = (bk.rand(*x.shape) >= self.p).astype(x.dtype) / (1.0 - self.p)
        return x * self.mask

    def backward(self, dy: Array) -> Array:
        return dy if self.mask is None else dy * self.mask


# Normalizaciones

class LayerNorm(Module):
    """Normaliza sobre el último eje."""

    def __init__(self, dim: int, eps: float = 1e-5) -> None:
        self.gamma = Parameter(xp.ones(dim), decay=False)
        self.beta = Parameter(xp.zeros(dim), decay=False)
        self.eps, self.dim = eps, dim

    def forward(self, x: Array) -> Array:
        assert x.shape[-1] == self.dim
        mu = x.mean(axis=-1, keepdims=True)
        xc = x - mu
        self.std = xp.sqrt((xc ** 2).mean(axis=-1, keepdims=True) + self.eps)
        self.xhat = xc / self.std
        return self.gamma.data * self.xhat + self.beta.data

    def backward(self, dy: Array) -> Array:
        ax = tuple(range(dy.ndim - 1))
        self.gamma.grad = (dy * self.xhat).sum(axis=ax)
        self.beta.grad = dy.sum(axis=ax)
        g = dy * self.gamma.data
        return (g - g.mean(axis=-1, keepdims=True)
                - self.xhat * (g * self.xhat).mean(axis=-1, keepdims=True)) / self.std


class BatchNorm2d(Module):
    """BatchNorm sobre (B, H, W) para tensores (B, C, H, W)."""

    def __init__(self, ch: int, eps: float = 1e-5, momentum: float = 0.1) -> None:
        self.gamma = Parameter(xp.ones(ch), decay=False)
        self.beta = Parameter(xp.zeros(ch), decay=False)
        self.run_mean = xp.zeros(ch, dtype=bk.DTYPE)
        self.run_var = xp.ones(ch, dtype=bk.DTYPE)
        self.eps, self.momentum, self.ch = eps, momentum, ch

    def forward(self, x: Array) -> Array:
        assert x.ndim == 4 and x.shape[1] == self.ch
        ax = (0, 2, 3)
        if self.training:
            n = x.shape[0] * x.shape[2] * x.shape[3]
            mu = x.mean(axis=ax)
            var = x.var(axis=ax)
            m = self.momentum
            self.run_mean = (1 - m) * self.run_mean + m * mu
            self.run_var = (1 - m) * self.run_var + m * var * (n / max(n - 1, 1))
        else:
            mu, var = self.run_mean, self.run_var
        self.std = xp.sqrt(var + self.eps)[None, :, None, None]
        self.xhat = (x - mu[None, :, None, None]) / self.std
        self.was_training = self.training
        return self.gamma.data[None, :, None, None] * self.xhat + self.beta.data[None, :, None, None]

    def backward(self, dy: Array) -> Array:
        ax = (0, 2, 3)
        self.gamma.grad = (dy * self.xhat).sum(axis=ax)
        self.beta.grad = dy.sum(axis=ax)
        g = dy * self.gamma.data[None, :, None, None]
        if not self.was_training:
            return g / self.std
        return (g - g.mean(axis=ax, keepdims=True)
                - self.xhat * (g * self.xhat).mean(axis=ax, keepdims=True)) / self.std


# Convolución y pooling

def _im2col(x: Array, kh: int, kw: int, s: int, p: int) -> Tuple[Array, int, int]:
    """(B,C,H,W) -> columnas (B, C*kh*kw, Ho*Wo)."""
    B, C, H, W = x.shape
    if p:
        x = xp.pad(x, ((0, 0), (0, 0), (p, p), (p, p)))
    Ho = (H + 2 * p - kh) // s + 1
    Wo = (W + 2 * p - kw) // s + 1
    cols = xp.empty((B, C, kh, kw, Ho, Wo), dtype=x.dtype)
    for i in range(kh):
        for j in range(kw):
            cols[:, :, i, j] = x[:, :, i:i + s * Ho:s, j:j + s * Wo:s]
    return cols.reshape(B, C * kh * kw, Ho * Wo), Ho, Wo


def _col2im(cols: Array, shape: Sequence[int], kh: int, kw: int, s: int, p: int,
            Ho: int, Wo: int) -> Array:
    """Adjunto exacto de _im2col (acumula las contribuciones solapadas)."""
    B, C, H, W = shape
    cols = cols.reshape(B, C, kh, kw, Ho, Wo)
    x = xp.zeros((B, C, H + 2 * p, W + 2 * p), dtype=cols.dtype)
    for i in range(kh):
        for j in range(kw):
            x[:, :, i:i + s * Ho:s, j:j + s * Wo:s] += cols[:, :, i, j]
    return x[:, :, p:H + p, p:W + p]


class Conv2d(Module):
    """Convolución 2D (correlación cruzada, igual que PyTorch) vía im2col."""

    def __init__(self, in_ch: int, out_ch: int, kernel_size: int = 3, stride: int = 1,
                 padding: int = 1, bias: bool = False) -> None:
        k = kernel_size
        std = math.sqrt(2.0 / (out_ch * k * k))
        self.W = Parameter(bk.randn(out_ch, in_ch, k, k) * std)
        self.b: Optional[Parameter] = Parameter(xp.zeros(out_ch), decay=False) if bias else None
        self.in_ch, self.out_ch, self.k, self.s, self.p = in_ch, out_ch, k, stride, padding

    def forward(self, x: Array) -> Array:
        assert x.ndim == 4 and x.shape[1] == self.in_ch, f"Conv2d esperaba C={self.in_ch}, llegó {x.shape}"
        self.x_shape = x.shape
        self.cols, self.Ho, self.Wo = _im2col(x, self.k, self.k, self.s, self.p)
        y = self.W.data.reshape(self.out_ch, -1) @ self.cols  # (B, O, Ho*Wo)
        if self.b is not None:
            y = y + self.b.data[None, :, None]
        return y.reshape(x.shape[0], self.out_ch, self.Ho, self.Wo)

    def backward(self, dy: Array) -> Array:
        B = dy.shape[0]
        d2 = dy.reshape(B, self.out_ch, -1)  # (B, O, L)
        self.W.grad = (d2 @ self.cols.transpose(0, 2, 1)).sum(axis=0).reshape(self.W.data.shape)
        if self.b is not None:
            self.b.grad = d2.sum(axis=(0, 2))
        dcols = self.W.data.reshape(self.out_ch, -1).T @ d2  # (B, C*k*k, L)
        return _col2im(dcols, self.x_shape, self.k, self.k, self.s, self.p, self.Ho, self.Wo)


class MaxPool2d(Module):
    """MaxPool kxk con stride k (H y W deben ser divisibles por k)."""

    def __init__(self, k: int = 2) -> None:
        self.k = k

    def forward(self, x: Array) -> Array:
        B, C, H, W = x.shape
        k = self.k
        assert H % k == 0 and W % k == 0, f"MaxPool2d: {H}x{W} no divisible por {k}"
        win = x.reshape(B, C, H // k, k, W // k, k).transpose(0, 1, 2, 4, 3, 5)
        win = win.reshape(B, C, H // k, W // k, k * k)
        self.idx = win.argmax(axis=-1)
        self.x_shape = x.shape
        return win.max(axis=-1)

    def backward(self, dy: Array) -> Array:
        B, C, H, W = self.x_shape
        k = self.k
        onehot = xp.arange(k * k)[None, None, None, None, :] == self.idx[..., None]
        g = onehot * dy[..., None]
        g = g.reshape(B, C, H // k, W // k, k, k).transpose(0, 1, 2, 4, 3, 5)
        return g.reshape(B, C, H, W).astype(dy.dtype)


class GlobalAvgPool2d(Module):
    """(B, C, H, W) -> (B, C)."""

    def forward(self, x: Array) -> Array:
        self.x_shape = x.shape
        return x.mean(axis=(2, 3))

    def backward(self, dy: Array) -> Array:
        B, C, H, W = self.x_shape
        return xp.broadcast_to(dy[:, :, None, None] / (H * W), self.x_shape).astype(dy.dtype)


def softmax(z: Array, axis: int = -1) -> Array:
    """Softmax estable (resta el máximo antes de exponenciar)."""
    e = xp.exp(z - z.max(axis=axis, keepdims=True))
    return e / e.sum(axis=axis, keepdims=True)


In [ ]:
%%writefile code_kaggle/losses.py
"""Focal Loss con su gradiente."""
from __future__ import annotations

from typing import Any, Optional, Tuple

from backend import xp

Array = Any


class FocalLoss:
    def __init__(self, gamma: float = 2.0, alpha: Optional[float] = 0.75) -> None:
        self.gamma, self.alpha = gamma, alpha

    def __call__(self, logits: Array, targets: Array) -> Tuple[float, Array]:
        """logits (B, K) y targets (B,) enteros -> (pérdida media, dL/dlogits de forma (B, K))."""
        B, K = logits.shape
        assert targets.shape == (B,), f"targets debe ser (B,), llegó {targets.shape}"
        z = logits - logits.max(axis=1, keepdims=True)
        logp = z - xp.log(xp.exp(z).sum(axis=1, keepdims=True))  # log-softmax estable
        p = xp.exp(logp)
        rows = xp.arange(B)
        u = logp[rows, targets]  # log p_t <= 0
        pt = xp.exp(u)
        one_m = xp.maximum(-xp.expm1(u), 1e-12)  # 1 - p_t sin cancelación
        if self.alpha is None:
            at = xp.ones_like(u)
        else:
            at = xp.where(targets == 1, self.alpha, 1.0 - self.alpha).astype(logits.dtype)
        loss = (-at * one_m ** self.gamma * u).mean()
        coef = at * (self.gamma * pt * one_m ** (self.gamma - 1.0) * u - one_m ** self.gamma)
        onehot = xp.zeros_like(logits)
        onehot[rows, targets] = 1.0
        dlogits = coef[:, None] * (onehot - p) / B
        return float(loss), dlogits


In [ ]:
%%writefile code_kaggle/optim.py
"""Optimizadores (SGD, AdamW), recorte de gradiente y tasa de aprendizaje coseno."""
from __future__ import annotations

import math
from typing import Any, List, Tuple

from backend import xp
from layers import Parameter


def clip_grad_norm(params: List[Parameter], max_norm: float = 5.0) -> float:
    """Recorte por norma global (como torch.nn.utils.clip_grad_norm_)."""
    total = math.sqrt(sum(float((p.grad ** 2).sum()) for p in params))
    if total > max_norm:
        scale = max_norm / (total + 1e-6)
        for p in params:
            p.grad *= scale
    return total


class SGD:
    """SGD con momento: v <- μ*v + g ; θ <- θ - lr*v."""

    def __init__(self, params: List[Parameter], lr: float = 1e-2, momentum: float = 0.9,
                 weight_decay: float = 0.0) -> None:
        self.params, self.lr, self.mu, self.wd = params, lr, momentum, weight_decay
        self.v = [xp.zeros_like(p.data) for p in params]

    def step(self) -> None:
        for p, v in zip(self.params, self.v):
            g = p.grad + self.wd * p.data if (self.wd and p.decay) else p.grad
            v *= self.mu
            v += g
            p.data -= (self.lr * p.lr_mult) * v


class AdamW:
    """AdamW (Loshchilov & Hutter): Adam con corrección de sesgo y weight decay desacoplado."""

    def __init__(self, params: List[Parameter], lr: float = 1e-4,
                 betas: Tuple[float, float] = (0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 1e-4) -> None:
        self.params, self.lr, self.eps, self.wd = params, lr, eps, weight_decay
        self.b1, self.b2 = betas
        self.m = [xp.zeros_like(p.data) for p in params]
        self.v = [xp.zeros_like(p.data) for p in params]
        self.t = 0

    def step(self) -> None:
        self.t += 1
        c1 = 1.0 - self.b1 ** self.t
        c2 = 1.0 - self.b2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            g = p.grad
            m *= self.b1
            m += (1.0 - self.b1) * g
            v *= self.b2
            v += (1.0 - self.b2) * g * g
            upd = (m / c1) / (xp.sqrt(v / c2) + self.eps)
            if self.wd and p.decay:
                upd = upd + self.wd * p.data
            p.data -= (self.lr * p.lr_mult) * upd


class CosineLR:
    """lr(e) = η_min + ½*(lr₀ - η_min)*(1 + cos(π*e/T_max))."""

    def __init__(self, optimizer: Any, t_max: int, eta_min: float = 1e-7) -> None:
        self.opt, self.t_max, self.eta_min = optimizer, t_max, eta_min
        self.base_lr = optimizer.lr
        self.epoch = 0

    def step(self) -> float:
        self.epoch += 1
        e = min(self.epoch, self.t_max)
        self.opt.lr = self.eta_min + 0.5 * (self.base_lr - self.eta_min) * (1 + math.cos(math.pi * e / self.t_max))
        return self.opt.lr


In [ ]:
%%writefile code_kaggle/mlp.py
"""Clasificador MLP de la etapa 3."""
from __future__ import annotations

from typing import Any

from layers import Dropout, Linear, Module, ReLU, Sequential

Array = Any


class MLPClassifier(Module):
    def __init__(self, in_dim: int = 512, hidden_dim: int = 256, num_classes: int = 2,
                 dropout: float = 0.3) -> None:
        self.in_dim, self.num_classes = in_dim, num_classes
        self.net = Sequential(
            Linear(in_dim, hidden_dim),
            ReLU(),
            Dropout(dropout),
            Linear(hidden_dim, num_classes),
        )

    def forward(self, x: Array) -> Array:
        assert x.ndim == 2 and x.shape[1] == self.in_dim, f"MLP esperaba (B, {self.in_dim}), llegó {x.shape}"
        return self.net(x)  # (B, num_classes)

    def backward(self, dlogits: Array) -> Array:
        return self.net.backward(dlogits)  # (B, in_dim)


In [ ]:
%%writefile code_kaggle/cnn.py
"""CNN de la rama lateral (mapa de asimetría 224x112 -> 256)."""
from __future__ import annotations

from typing import Any, Tuple

from layers import (BatchNorm2d, Conv2d, GlobalAvgPool2d, LayerNorm, Linear, MaxPool2d,
                    Module, ReLU, Sequential)

Array = Any


def conv_block(in_ch: int, out_ch: int) -> Sequential:
    return Sequential(
        Conv2d(in_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False),
        BatchNorm2d(out_ch),
        ReLU(),
        MaxPool2d(2),
    )


class CNNBranch(Module):
    def __init__(self, in_channels: int = 3, widths: Tuple[int, int, int] = (32, 64, 128),
                 proj_dim: int = 256) -> None:
        self.in_channels = in_channels
        chans = (in_channels,) + tuple(widths)
        self.blocks = Sequential(*[conv_block(chans[i], chans[i + 1]) for i in range(3)])
        self.pool = GlobalAvgPool2d()
        self.proj = Sequential(Linear(widths[-1], proj_dim), LayerNorm(proj_dim))

    def forward(self, x: Array) -> Array:
        assert x.ndim == 4 and x.shape[1] == self.in_channels, f"CNN esperaba (B,{self.in_channels},H,W), llegó {x.shape}"
        assert x.shape[2] % 8 == 0 and x.shape[3] % 8 == 0, "H y W deben ser divisibles por 8 (3 MaxPool)"
        f = self.blocks(x)  # (B, 128, H/8, W/8)
        v = self.pool(f)  # (B, 128)
        return self.proj(v)  # (B, 256)

    def backward(self, dy: Array) -> Array:
        d = self.proj.backward(dy)  # (B, 128)
        d = self.pool.backward(d)  # (B, 128, H/8, W/8)
        return self.blocks.backward(d)  # (B, 3, H, W)


In [ ]:
%%writefile code_kaggle/swin.py
"""Swin Transformer (Swin-T) implementado desde cero: rama global 224x224 -> 768 -> 256."""
from __future__ import annotations

from typing import Any, List, Optional, Sequence, Tuple

import numpy as np

import backend as bk
from backend import xp
from layers import GELU, LayerNorm, Linear, Module, Parameter, Sequential, softmax

Array = Any
MASK_VALUE = -100.0


# utilidades de ventanas
def window_partition(x: Array, w: int) -> Array:
    """(B, H, W, C) -> (B*nW, w*w, C) con nW = (H/w)*(W/w)."""
    B, H, W, C = x.shape
    x = x.reshape(B, H // w, w, W // w, w, C).transpose(0, 1, 3, 2, 4, 5)
    return x.reshape(-1, w * w, C)


def window_reverse(win: Array, w: int, B: int, H: int, W: int) -> Array:
    """(B*nW, w*w, C) -> (B, H, W, C)."""
    C = win.shape[-1]
    x = win.reshape(B, H // w, W // w, w, w, C).transpose(0, 1, 3, 2, 4, 5)
    return x.reshape(B, H, W, C)


def relative_position_index(w: int) -> np.ndarray:
    """Índice (w^2, w^2) a la tabla de sesgos ((2w-1)^2, h)."""
    coords = np.stack(np.meshgrid(np.arange(w), np.arange(w), indexing="ij")).reshape(2, -1)  # (2, N)
    rel = coords[:, :, None] - coords[:, None, :] + (w - 1)  # (2, N, N)
    return rel[0] * (2 * w - 1) + rel[1]


def shifted_window_mask(H: int, W: int, w: int, s: int) -> np.ndarray:
    """Máscara (nW, N, N): 0 si dos tokens provienen de la misma región antes del roll, -100 si no."""
    img = np.zeros((1, H, W, 1))
    cnt = 0
    for hs in (slice(0, -w), slice(-w, -s), slice(-s, None)):
        for ws in (slice(0, -w), slice(-w, -s), slice(-s, None)):
            img[:, hs, ws, :] = cnt
            cnt += 1
    mw = window_partition(img, w)[..., 0]  # (nW, N)
    diff = mw[:, None, :] - mw[:, :, None]
    return np.where(diff != 0, MASK_VALUE, 0.0)


# capas Swin
class PatchEmbedSwin(Module):
    """Conv(k=p, s=p) implementada como Linear sobre parches + LayerNorm."""

    def __init__(self, img_size: int = 224, patch_size: int = 4, in_chans: int = 3,
                 embed_dim: int = 96) -> None:
        assert img_size % patch_size == 0, "img_size debe ser divisible por patch_size"
        self.img_size, self.p, self.c = img_size, patch_size, in_chans
        self.grid = img_size // patch_size
        self.proj = Linear(in_chans * patch_size ** 2, embed_dim, init="trunc")
        self.norm = LayerNorm(embed_dim)

    def forward(self, x: Array) -> Array:
        B, C, H, W = x.shape
        assert (C, H, W) == (self.c, self.img_size, self.img_size), \
            f"PatchEmbedSwin esperaba (B,{self.c},{self.img_size},{self.img_size}), llegó {x.shape}"
        p, g = self.p, self.grid
        patches = x.reshape(B, C, g, p, g, p).transpose(0, 2, 4, 1, 3, 5).reshape(B, g * g, C * p * p)
        return self.norm(self.proj(patches))

    def backward(self, dy: Array) -> Array:
        d = self.proj.backward(self.norm.backward(dy))
        B, p, g, C = d.shape[0], self.p, self.grid, self.c
        return d.reshape(B, g, g, C, p, p).transpose(0, 3, 1, 4, 2, 5).reshape(B, C, g * p, g * p)


class WindowAttention(Module):
    """MSA dentro de cada ventana wxw con sesgo de posición relativa aprendido."""

    def __init__(self, dim: int, window: int, num_heads: int) -> None:
        assert dim % num_heads == 0, f"dim={dim} debe ser divisible por num_heads={num_heads}"
        self.dim, self.w, self.h, self.dh = dim, window, num_heads, dim // num_heads
        self.N = window * window
        self.scale = self.dh ** -0.5
        self.qkv = Linear(dim, 3 * dim, init="trunc")
        self.proj = Linear(dim, dim, init="trunc")
        T = (2 * window - 1) ** 2
        self.bias_table = Parameter(xp.clip(bk.randn(T, num_heads) * 0.02, -0.04, 0.04), decay=False)
        idx = relative_position_index(window).reshape(-1)  # (N^2,)
        onehot = np.zeros((idx.size, T))
        onehot[np.arange(idx.size), idx] = 1.0
        self._idx = xp.asarray(idx)
        self._P = bk.asarray(onehot)  # (N^2, T) constante

    def forward(self, x: Array, mask: Optional[Array] = None) -> Array:
        Bw, N, C = x.shape
        assert N == self.N and C == self.dim, f"WindowAttention esperaba (*, {self.N}, {self.dim}), llegó {x.shape}"
        qkv = self.qkv(x).reshape(Bw, N, 3, self.h, self.dh).transpose(2, 0, 3, 1, 4)  # (3,Bw,h,N,dh)
        self.q, self.k, self.v = qkv[0], qkv[1], qkv[2]
        s = (self.q @ self.k.transpose(0, 1, 3, 2)) * self.scale  # (Bw,h,N,N)
        rel = self.bias_table.data[self._idx].reshape(N, N, self.h).transpose(2, 0, 1)  # (h,N,N)
        s = s + rel[None]
        if mask is not None:
            nW = mask.shape[0]
            s = (s.reshape(Bw // nW, nW, self.h, N, N) + mask[None, :, None]).reshape(Bw, self.h, N, N)
        self.attn = softmax(s, axis=-1)
        out = (self.attn @ self.v).transpose(0, 2, 1, 3).reshape(Bw, N, C)
        return self.proj(out)

    def backward(self, dy: Array) -> Array:
        Bw, N, C = dy.shape
        do = self.proj.backward(dy).reshape(Bw, N, self.h, self.dh).transpose(0, 2, 1, 3)
        da = do @ self.v.transpose(0, 1, 3, 2)
        dv = self.attn.transpose(0, 1, 3, 2) @ do
        ds = self.attn * (da - (da * self.attn).sum(axis=-1, keepdims=True))  # (Bw,h,N,N)
        drel = ds.sum(axis=0).reshape(self.h, N * N)  # (h, N^2)
        self.bias_table.grad = self._P.T @ drel.T  # (T, h)
        dq = (ds @ self.k) * self.scale
        dk = (ds.transpose(0, 1, 3, 2) @ self.q) * self.scale
        dqkv = xp.stack([dq, dk, dv], axis=0).transpose(1, 3, 0, 2, 4).reshape(Bw, N, 3 * C)
        return self.qkv.backward(dqkv)


class SwinBlock(Module):
    """x <- x + Rev(SW/W-MSA(Part(roll(LN(x))))) ; x <- x + MLP(LN(x))."""

    def __init__(self, dim: int, resolution: Tuple[int, int], num_heads: int, window: int = 7,
                 shift: int = 0, mlp_ratio: float = 4.0) -> None:
        H, W = resolution
        if min(H, W) <= window:  # la ventana cubre toda la imagen: sin desplazamiento
            window, shift = min(H, W), 0
        assert H % window == 0 and W % window == 0, f"resolución {resolution} no divisible por ventana {window}"
        assert 0 <= shift < window
        self.dim, self.H, self.W, self.w, self.s = dim, H, W, window, shift
        self.norm1 = LayerNorm(dim)
        self.attn = WindowAttention(dim, window, num_heads)
        self.norm2 = LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = Sequential(Linear(dim, hidden, init="trunc"), GELU(), Linear(hidden, dim, init="trunc"))
        self.mask = bk.asarray(shifted_window_mask(H, W, window, shift)) if shift > 0 else None

    def forward(self, x: Array) -> Array:
        B, L, C = x.shape
        assert L == self.H * self.W and C == self.dim, f"SwinBlock esperaba (B,{self.H * self.W},{self.dim}), llegó {x.shape}"
        h = self.norm1(x).reshape(B, self.H, self.W, C)
        if self.s:
            h = xp.roll(h, (-self.s, -self.s), axis=(1, 2))
        h = self.attn(window_partition(h, self.w), self.mask)
        h = window_reverse(h, self.w, B, self.H, self.W)
        if self.s:
            h = xp.roll(h, (self.s, self.s), axis=(1, 2))
        x = x + h.reshape(B, L, C)
        return x + self.mlp(self.norm2(x))

    def backward(self, dy: Array) -> Array:
        B, L, C = dy.shape
        d = dy + self.norm2.backward(self.mlp.backward(dy))
        g = d.reshape(B, self.H, self.W, C)
        if self.s:  # adjunta de roll(+s) = roll(-s)
            g = xp.roll(g, (-self.s, -self.s), axis=(1, 2))
        g = self.attn.backward(window_partition(g, self.w))  # adjunta de reverse = partition
        g = window_reverse(g, self.w, B, self.H, self.W)
        if self.s:
            g = xp.roll(g, (self.s, self.s), axis=(1, 2))
        return d + self.norm1.backward(g.reshape(B, L, C))


class PatchMerging(Module):
    """(B, H*W, C) -> concat de vecinos 2x2 (B, H/2*W/2, 4C) -> LayerNorm -> Linear(4C, 2C, sin sesgo)."""

    def __init__(self, resolution: Tuple[int, int], dim: int) -> None:
        H, W = resolution
        assert H % 2 == 0 and W % 2 == 0, f"PatchMerging requiere resolución par, llegó {resolution}"
        self.H, self.W, self.dim = H, W, dim
        self.norm = LayerNorm(4 * dim)
        self.red = Linear(4 * dim, 2 * dim, bias=False, init="trunc")

    def forward(self, x: Array) -> Array:
        B, L, C = x.shape
        assert L == self.H * self.W and C == self.dim
        # orden de concatenación oficial: x0=(0,0), x1=(1,0), x2=(0,1), x3=(1,1)
        x = x.reshape(B, self.H // 2, 2, self.W // 2, 2, C).transpose(0, 1, 3, 4, 2, 5)
        x = x.reshape(B, (self.H // 2) * (self.W // 2), 4 * C)
        return self.red(self.norm(x))

    def backward(self, dy: Array) -> Array:
        d = self.norm.backward(self.red.backward(dy))
        B, C = d.shape[0], self.dim
        d = d.reshape(B, self.H // 2, self.W // 2, 2, 2, C).transpose(0, 1, 4, 2, 3, 5)
        return d.reshape(B, self.H * self.W, C)


class SwinBranch(Module):
    def __init__(self, img_size: int = 224, patch_size: int = 4, in_chans: int = 3,
                 embed_dim: int = 96, depths: Sequence[int] = (2, 2, 6, 2),
                 num_heads: Sequence[int] = (3, 6, 12, 24), window: int = 7,
                 mlp_ratio: float = 4.0, proj_dim: int = 256) -> None:
        assert len(depths) == len(num_heads)
        self.patch_embed = PatchEmbedSwin(img_size, patch_size, in_chans, embed_dim)
        res = self.patch_embed.grid
        self.layers: List[Module] = []
        dim = embed_dim
        for i, (dep, nh) in enumerate(zip(depths, num_heads)):
            for j in range(dep):
                self.layers.append(SwinBlock(dim, (res, res), nh, window,
                                             shift=0 if j % 2 == 0 else window // 2, mlp_ratio=mlp_ratio))
            if i < len(depths) - 1:
                self.layers.append(PatchMerging((res, res), dim))
                res, dim = res // 2, dim * 2
        self.norm = LayerNorm(dim)
        self.proj = Sequential(Linear(dim, proj_dim), LayerNorm(proj_dim))
        self.embed_dim, self.num_features, self.proj_dim = embed_dim, dim, proj_dim

    def forward_features(self, x: Array) -> Array:
        """(B, 3, H, W) -> vector global (B, num_features) = (B, 768) en Swin-T."""
        x = self.patch_embed(x)
        for lyr in self.layers:
            x = lyr(x)
        x = self.norm(x)
        self.n_tokens = x.shape[1]
        return x.mean(axis=1)

    def forward(self, x: Array) -> Array:
        return self.proj(self.forward_features(x))  # (B, 256)

    def backward(self, dy: Array) -> Array:
        d = self.proj.backward(dy)
        d = xp.broadcast_to(d[:, None, :] / self.n_tokens, (d.shape[0], self.n_tokens, d.shape[1]))
        d = self.norm.backward(xp.ascontiguousarray(d))
        for lyr in reversed(self.layers):
            d = lyr.backward(d)
        return self.patch_embed.backward(d)


In [ ]:
%%writefile code_kaggle/pretrained.py
"""Carga de pesos ImageNet de Swin-T (formato safetensors) sin PyTorch."""
from __future__ import annotations

import json
import os
import re
import struct
import urllib.request
from typing import Dict, Optional

import numpy as np

from layers import LayerNorm, Linear
from swin import PatchMerging, SwinBlock, SwinBranch

SWIN_T_URL = ("https://huggingface.co/timm/swin_tiny_patch4_window7_224.ms_in1k/"
              "resolve/main/model.safetensors")

_DT = {"F32": np.float32, "F16": np.float16, "F64": np.float64, "I64": np.int64, "I32": np.int32}


def read_safetensors(path: str) -> Dict[str, np.ndarray]:
    """Lector mínimo del formato safetensors: u64 little-endian (n) + JSON (n bytes) + datos."""
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        header = json.loads(f.read(n))
        blob = f.read()
    out = {}
    for k, meta in header.items():
        if k == "__metadata__":
            continue
        a, b = meta["data_offsets"]
        raw = blob[a:b]
        if meta["dtype"] == "BF16":  # bfloat16 = 16 bits altos de un float32
            arr = (np.frombuffer(raw, dtype="<u2").astype(np.uint32) << 16).view(np.float32)
        else:
            arr = np.frombuffer(raw, dtype=np.dtype(_DT[meta["dtype"]]).newbyteorder("<"))
        out[k] = arr.reshape(meta["shape"]).astype(np.float32) if arr.dtype.kind == "f" else arr.reshape(meta["shape"])
    return out


def download(url: str = SWIN_T_URL, dst: str = "swin_tiny_imagenet.safetensors") -> str:
    if not os.path.exists(dst):
        tmp = dst + ".part"
        urllib.request.urlretrieve(url, tmp)
        os.replace(tmp, dst)
    return dst


def _normalize_keys(sd: Dict[str, np.ndarray]) -> Dict[str, np.ndarray]:
    """Pasa los nombres de timm >= 0.9 (downsample al inicio de la etapa) al formato antiguo."""
    old_style = any(k.startswith("layers.0.downsample") for k in sd)
    out = {}
    for k, v in sd.items():
        if k.startswith("head") or "relative_position_index" in k or "attn_mask" in k:
            continue
        m = re.match(r"layers\.(\d+)\.downsample\.(.*)", k)
        if m and old_style:
            k = f"layers.{int(m.group(1)) + 1}.downsample.{m.group(2)}"
        out[k] = v
    return out


def _set(dst, src: np.ndarray, name: str, xp) -> None:
    assert tuple(dst.data.shape) == tuple(src.shape), f"{name}: {src.shape} ≠ {dst.data.shape}"
    dst.data[...] = xp.asarray(src, dtype=dst.data.dtype)


def load_swin_imagenet(model: SwinBranch, path: Optional[str] = None, verbose: bool = True) -> int:
    """Copia los pesos ImageNet en `model` (Swin-T: embed 96, profundidades 2-2-6-2, ventana 7)."""
    from backend import xp

    sd = _normalize_keys(read_safetensors(path or download()))
    used = set()

    def take(k: str) -> np.ndarray:
        used.add(k)
        return sd[k]

    def ln(dst: LayerNorm, pre: str) -> None:
        _set(dst.gamma, take(pre + ".weight"), pre, xp)
        _set(dst.beta, take(pre + ".bias"), pre, xp)

    def lin(dst: Linear, pre: str) -> None:
        _set(dst.W, take(pre + ".weight").T, pre, xp)
        if dst.b is not None:
            _set(dst.b, take(pre + ".bias"), pre, xp)

    pe = model.patch_embed
    w = take("patch_embed.proj.weight")  # (96, 3, 4, 4) -> (3*4*4, 96)
    _set(pe.proj.W, w.reshape(w.shape[0], -1).T, "patch_embed.proj", xp)
    _set(pe.proj.b, take("patch_embed.proj.bias"), "patch_embed.proj", xp)
    ln(pe.norm, "patch_embed.norm")

    stage, block = 0, 0
    for lyr in model.layers:
        if isinstance(lyr, PatchMerging):
            stage, block = stage + 1, 0
            pre = f"layers.{stage}.downsample"
            ln(lyr.norm, pre + ".norm")
            lin(lyr.red, pre + ".reduction")
            continue
        assert isinstance(lyr, SwinBlock)
        pre = f"layers.{stage}.blocks.{block}"
        ln(lyr.norm1, pre + ".norm1")
        lin(lyr.attn.qkv, pre + ".attn.qkv")
        lin(lyr.attn.proj, pre + ".attn.proj")
        _set(lyr.attn.bias_table, take(pre + ".attn.relative_position_bias_table"), pre, xp)
        ln(lyr.norm2, pre + ".norm2")
        lin(lyr.mlp.mods[0], pre + ".mlp.fc1")
        lin(lyr.mlp.mods[2], pre + ".mlp.fc2")
        block += 1
    ln(model.norm, "norm")

    missing = sorted(set(sd) - used)
    assert not missing, f"Pesos del checkpoint sin usar (arquitectura distinta): {missing[:5]}"
    if verbose:
        print(f"Swin-T ImageNet-1k: {len(used)} tensores copiados "
              f"({sum(int(np.prod(sd[k].shape)) for k in used):,} parámetros)")
    return len(used)


def set_lr_mult(model, mult: float) -> None:
    """Multiplicador de tasa de aprendizaje para todos los parámetros de `model` (p."""
    for p in model.parameters():
        p.lr_mult = mult


In [ ]:
%%writefile code_kaggle/detector.py
"""Detector completo: rama Swin + rama de asimetría + MLP."""
from __future__ import annotations

import random
from typing import Any, Optional

import backend as bk
from backend import xp
from cnn import CNNBranch
from layers import Module, softmax
from mlp import MLPClassifier
from swin import SwinBranch

Array = Any


class DeepfakeDetectorScratch(Module):
    def __init__(self, swin: Optional[SwinBranch] = None, cnn: Optional[CNNBranch] = None,
                 mlp: Optional[MLPClassifier] = None, drop_branch: float = 0.1,
                 branches: str = "ambas") -> None:
        """branches: "ambas" (pipeline completo) | "solo_swin" | "solo_asimetria" (ablaciones)."""
        assert branches in ("ambas", "solo_swin", "solo_asimetria")
        self.branches = branches
        self.swin = swin if swin is not None else SwinBranch()
        self.cnn = cnn if cnn is not None else CNNBranch()
        self.mlp = mlp if mlp is not None else MLPClassifier()
        self.drop_branch = drop_branch
        self._keep_g, self._keep_l = 1.0, 1.0

    def forward(self, face: Array, diff_map: Array, lateral_mask: Optional[Array] = None) -> Array:
        """lateral_mask (B,) con 0 o 1: en 0 se apaga la rama lateral de ese frame (cabeza muy girada)."""
        face, diff_map = bk.asarray(face), bk.asarray(diff_map)
        assert face.shape[0] == diff_map.shape[0], "face y diff_map deben tener el mismo B"
        B = face.shape[0]
        fg = self.swin(face) if self.branches != "solo_asimetria" else xp.zeros((B, self.mlp_half()), dtype=bk.DTYPE)
        fl = self.cnn(diff_map) if self.branches != "solo_swin" else xp.zeros((B, self.mlp_half()), dtype=bk.DTYPE)
        self._keep_g, self._keep_l = 1.0, 1.0
        if self.training and self.drop_branch > 0 and self.branches == "ambas":
            r = random.random()
            if r < self.drop_branch / 2:
                self._keep_g = 0.0
            elif r < self.drop_branch:
                self._keep_l = 0.0
        self._d = fg.shape[1]
        self._lmask = None if lateral_mask is None else bk.asarray(lateral_mask).reshape(-1, 1).astype(fl.dtype)
        if self._lmask is not None:
            assert self._lmask.shape[0] == B, "lateral_mask debe tener B elementos"
            fl = fl * self._lmask
        fused = xp.concatenate([fg * self._keep_g, fl * self._keep_l], axis=1)  # (B, 512)
        return self.mlp(fused)  # (B, 2)

    def backward(self, dlogits: Array) -> None:
        dfused = self.mlp.backward(dlogits)  # (B, 512)
        if self.branches != "solo_asimetria":
            self.swin.backward(dfused[:, :self._d] * self._keep_g)
        if self.branches != "solo_swin":
            dl = dfused[:, self._d:] * self._keep_l
            self.cnn.backward(dl if self._lmask is None else dl * self._lmask)

    def mlp_half(self) -> int:
        return self.swin.proj_dim

    def predict_video(self, faces: Array, diff_maps: Array, chunk: int = 16,
                      lateral_mask: Optional[Array] = None) -> float:
        """faces (N,3,224,224), diff_maps (N,3,224,112) -> P(deepfake) promedio del video."""
        was_training = self.training
        self.eval()
        probs = []
        for i in range(0, faces.shape[0], chunk):
            lm = None if lateral_mask is None else lateral_mask[i:i + chunk]
            logits = self.forward(faces[i:i + chunk], diff_maps[i:i + chunk], lm)
            probs.append(softmax(logits, axis=1)[:, 1])
            self.release()
        self.train(was_training)
        return float(xp.concatenate(probs).mean())


In [ ]:
%%writefile code_kaggle/augment.py
"""Degradaciones de calidad, aumento de datos, normalización robusta y mapa de asimetría."""
from __future__ import annotations

import math
from dataclasses import dataclass, field
from functools import lru_cache
from typing import Any, Dict, Optional, Tuple

import numpy as np

import backend as bk
from backend import xp

Array = Any


# Operadores básicos

@lru_cache(maxsize=64)
def _gauss_kernel_np(sigma: float) -> np.ndarray:
    r = max(1, int(math.ceil(3.0 * sigma)))
    t = np.arange(-r, r + 1, dtype=np.float64)
    k = np.exp(-(t ** 2) / (2.0 * sigma ** 2))
    return k / k.sum()


def gaussian_blur(x: Array, sigma: float) -> Array:
    """Desenfoque gaussiano separable con borde reflejado."""
    if sigma <= 0:
        return x
    k = xp.asarray(_gauss_kernel_np(round(float(sigma), 3)), dtype=x.dtype)
    r = (k.shape[0] - 1) // 2
    H, W = x.shape[-2], x.shape[-1]
    assert r < min(H, W), "sigma demasiado grande para el tamaño de la imagen"
    pad = [(0, 0)] * (x.ndim - 2)
    xh = xp.pad(x, pad + [(0, 0), (r, r)], mode="reflect")
    y = sum(k[i] * xh[..., :, i:i + W] for i in range(2 * r + 1))
    yv = xp.pad(y, pad + [(r, r), (0, 0)], mode="reflect")
    return sum(k[i] * yv[..., i:i + H, :] for i in range(2 * r + 1))


@lru_cache(maxsize=64)
def _bilinear_matrix_np(n_in: int, n_out: int) -> np.ndarray:
    """Matriz (n_out, n_in) de interpolación lineal con centros de píxel (como cv2.INTER_LINEAR)."""
    src = (np.arange(n_out) + 0.5) * (n_in / n_out) - 0.5
    src = np.clip(src, 0.0, n_in - 1.0)
    i0 = np.floor(src).astype(np.int64)
    i1 = np.minimum(i0 + 1, n_in - 1)
    w1 = src - i0
    M = np.zeros((n_out, n_in))
    rows = np.arange(n_out)
    np.add.at(M, (rows, i0), 1.0 - w1)
    np.add.at(M, (rows, i1), w1)
    return M


@lru_cache(maxsize=64)
def _area_matrix_np(n_in: int, n_out: int) -> np.ndarray:
    """Matriz (n_out, n_in) de submuestreo por promedio de área (antialias, como cv2.INTER_AREA)."""
    s = n_in / n_out
    M = np.zeros((n_out, n_in))
    for o in range(n_out):
        a, b = o * s, (o + 1) * s
        for i in range(int(math.floor(a)), min(int(math.ceil(b)), n_in)):
            M[o, i] = (min(b, i + 1) - max(a, i)) / s
    return M


def resize(x: Array, h_out: int, w_out: int, mode: str = "bilinear") -> Array:
    """Redimensiona (..., H, W) -> (..., h_out, w_out) como y = M_h * x * M_w^T."""
    H, W = x.shape[-2], x.shape[-1]
    f = _area_matrix_np if mode == "area" else _bilinear_matrix_np
    Mh = xp.asarray(f(H, h_out), dtype=x.dtype)
    Mw = xp.asarray(f(W, w_out), dtype=x.dtype)
    return xp.matmul(xp.matmul(Mh, x), Mw.T)


def downscale_upscale(x: Array, factor: float) -> Array:
    """Reduce la resolución por `factor` (0 < factor <= 1) y vuelve al tamaño original."""
    if factor >= 1.0:
        return x
    H, W = x.shape[-2], x.shape[-1]
    h, w = max(8, int(round(H * factor))), max(8, int(round(W * factor)))
    return resize(resize(x, h, w, "area"), H, W, "bilinear")


def add_gaussian_noise(x: Array, std: float, rng: Any = None) -> Array:
    if std <= 0:
        return x
    r = rng if rng is not None else xp.random
    return x + std * r.standard_normal(x.shape).astype(x.dtype)


def quantize_uint8(x: Array) -> Array:
    """Recorta a [0,1] y cuantiza a 256 niveles (como una imagen guardada en 8 bits)."""
    return xp.round(xp.clip(x, 0.0, 1.0) * 255.0) / 255.0


def jpeg_compress(x: Array, quality: int) -> Array:
    """Compresión JPEG real con OpenCV (solo evaluación, degradación fuera de distribución)."""
    import cv2

    arr = bk.to_numpy(xp.clip(x, 0, 1))
    out = np.empty_like(arr)
    for b in range(arr.shape[0]):
        img = (arr[b].transpose(1, 2, 0) * 255.0 + 0.5).astype(np.uint8)
        ok, buf = cv2.imencode(".jpg", img[:, :, ::-1], [int(cv2.IMWRITE_JPEG_QUALITY), int(quality)])
        assert ok, "Falló la codificación JPEG"
        dec = cv2.imdecode(buf, cv2.IMREAD_COLOR)[:, :, ::-1]
        out[b] = dec.transpose(2, 0, 1).astype(np.float32) / 255.0
    return bk.asarray(out)


DIFF_MODES = ("pixel", "residual")


def bilateral_diff_map(face: Array, mode: str = "pixel", sigma: float = 1.0) -> Array:
    """Rama lateral: (B,3,H,W) en [0,1] -> mapa de asimetría (B,3,H,W/2) = |fliplr(izq) - der|."""
    assert mode in DIFF_MODES, f"mode debe ser uno de {DIFF_MODES}"
    W = face.shape[-1]
    assert W % 2 == 0, "El ancho del rostro debe ser par para dividir en hemicaras"
    half = W // 2
    if mode == "residual":
        face = face - gaussian_blur(face, sigma)
    left, right = face[..., :half][..., ::-1], face[..., half:]
    if mode == "residual":
        eps = 1e-6
        left = left / (xp.sqrt((left * left).mean(axis=(-2, -1), keepdims=True)) + eps)
        right = right / (xp.sqrt((right * right).mean(axis=(-2, -1), keepdims=True)) + eps)
    return xp.abs(left - right)


# Degradación determinista (evaluación)

# Niveles de calidad para la evaluación comparativa. 'buena' = imagen original.
QUALITY_LEVELS: Dict[str, Dict[str, float]] = {
    "buena":    {},
    "regular":  {"down": 0.75, "blur": 0.8, "noise": 3 / 255},
    "mala":     {"down": 0.50, "blur": 1.2, "noise": 6 / 255},
    "muy_mala": {"down": 0.30, "blur": 2.0, "noise": 12 / 255},
}

# Degradaciones aisladas (diagnóstico) y una no vista en entrenamiento (JPEG).
ISOLATED_DEGRADATIONS: Dict[str, Dict[str, float]] = {
    "solo_desenfoque": {"blur": 1.5},
    "solo_ruido":      {"noise": 10 / 255},
    "solo_baja_res":   {"down": 0.35},
    "jpeg_q30_no_visto": {"jpeg": 30},
}


def apply_degradation(x: Array, spec: Dict[str, float], seed: int = 0) -> Array:
    """Aplica una degradación fija y reproducible."""
    if not spec:
        return x
    rng = xp.random.RandomState(seed)
    if spec.get("down", 1.0) < 1.0:
        x = downscale_upscale(x, spec["down"])
    if spec.get("blur", 0.0) > 0:
        x = gaussian_blur(x, spec["blur"])
    if spec.get("noise", 0.0) > 0:
        x = add_gaussian_noise(x, spec["noise"], rng)
    x = quantize_uint8(x)
    if spec.get("jpeg"):
        x = jpeg_compress(x, int(spec["jpeg"]))
    return x


# Aumento aleatorio (entrenamiento)

@dataclass
class AugConfig:
    """Probabilidades y rangos del aumento de calidad (muestreados por imagen)."""
    p_down: float = 0.5
    down_range: Tuple[float, float] = (0.25, 0.9)
    p_blur: float = 0.5
    sigma_range: Tuple[float, float] = (0.4, 2.2)
    p_noise: float = 0.5
    noise_range: Tuple[float, float] = (1 / 255, 15 / 255)
    p_color: float = 0.3
    brightness: float = 0.08
    contrast: float = 0.15
    # Sin flip horizontal: invertiría las hemicaras del mapa diferencial.


class QualityAugmenter:
    """Aumento de datos de calidad (desenfoque, ruido, baja resolución, brillo/contraste)."""

    def __init__(self, cfg: Optional[AugConfig] = None) -> None:
        self.cfg = cfg or AugConfig()

    def sample_params(self) -> Dict[str, float]:
        c, r = self.cfg, np.random
        p: Dict[str, float] = {}
        if r.rand() < c.p_down:
            p["down"] = float(r.uniform(*c.down_range))
        if r.rand() < c.p_blur:
            p["blur"] = float(r.uniform(*c.sigma_range))
        if r.rand() < c.p_noise:
            p["noise"] = float(r.uniform(*c.noise_range))
        if r.rand() < c.p_color:
            p["bright"] = float(r.uniform(-c.brightness, c.brightness))
            p["contrast"] = float(r.uniform(1 - c.contrast, 1 + c.contrast))
        return p

    def __call__(self, x: Array) -> Array:
        out = xp.empty_like(x)
        for b in range(x.shape[0]):
            p = self.sample_params()
            img = x[b:b + 1]
            if "down" in p:
                img = downscale_upscale(img, p["down"])
            if "blur" in p:
                img = gaussian_blur(img, p["blur"])
            if "contrast" in p:
                m = img.mean(axis=(-2, -1), keepdims=True)
                img = (img - m) * p["contrast"] + m + p["bright"]
            if "noise" in p:
                img = add_gaussian_noise(img, p["noise"])
            out[b:b + 1] = quantize_uint8(img)
        return out


# Normalización robusta

class RobustNormalizer:
    """Normalización por canal con estadísticos robustos: z = clip((x - mediana) / (IQR/1.349), ±c)."""

    def __init__(self, clip: float = 5.0) -> None:
        self.clip = clip
        self.center: Optional[np.ndarray] = None
        self.scale: Optional[np.ndarray] = None

    def fit(self, x: np.ndarray, max_pixels: int = 2_000_000, seed: int = 42) -> "RobustNormalizer":
        """x: np.ndarray (N, C, H, W) en [0,1] del conjunto de entrenamiento."""
        x = np.asarray(x)
        C = x.shape[1]
        flat = x.transpose(1, 0, 2, 3).reshape(C, -1)
        if flat.shape[1] > max_pixels:
            idx = np.random.RandomState(seed).choice(flat.shape[1], max_pixels, replace=False)
            flat = flat[:, idx]
        q25, q50, q75 = np.percentile(flat, [25, 50, 75], axis=1)
        self.center = q50.astype(np.float32)
        self.scale = np.maximum((q75 - q25) / 1.349, 1e-3).astype(np.float32)
        return self

    def __call__(self, x: Array) -> Array:
        assert self.center is not None, "RobustNormalizer sin ajustar: llama a fit() con train"
        c = xp.asarray(self.center, dtype=x.dtype)[None, :, None, None]
        s = xp.asarray(self.scale, dtype=x.dtype)[None, :, None, None]
        return xp.clip((x - c) / s, -self.clip, self.clip)

    def to_dict(self) -> Dict[str, Any]:
        return {"center": self.center.tolist(), "scale": self.scale.tolist(), "clip": self.clip}

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "RobustNormalizer":
        n = cls(d["clip"])
        n.center = np.asarray(d["center"], dtype=np.float32)
        n.scale = np.asarray(d["scale"], dtype=np.float32)
        return n


In [ ]:
%%writefile code_kaggle/data.py
"""Lectura de videos, extracción y alineación de rostros, cachés y particiones."""
from __future__ import annotations

import os
import time
import urllib.request
from collections import defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import cv2
import numpy as np

VIDEO_EXT = (".mp4", ".avi", ".mov", ".mkv")
LABELS = {"real": 0, "fake": 1}


@dataclass
class VideoItem:
    path: str
    label: int
    method: str
    video_id: str
    identities: Tuple[str, ...]
    group: int = -1
    split: str = ""


def parse_name(stem: str, label: int) -> Tuple[str, Tuple[str, ...]]:
    parts = stem.split("_")
    if label == 0 and stem.isdigit():
        return "real", (stem,)
    if len(parts) >= 3 and parts[-1].isdigit() and parts[-2].isdigit():
        return "_".join(parts[:-2]), (parts[-2], parts[-1])
    return ("real" if label == 0 else "fake"), (stem,)


def list_videos(root: str) -> List[VideoItem]:
    items: List[VideoItem] = []
    for name, lab in LABELS.items():
        d = os.path.join(root, name)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VIDEO_EXT):
                stem = os.path.splitext(f)[0]
                method, ids = parse_name(stem, lab)
                items.append(VideoItem(os.path.join(d, f), lab, method, stem, ids))
    if not items:
        raise FileNotFoundError(f"No hay videos en {root}/real ni {root}/fake")
    return items


FFPP_METHODS = ("Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures")  # protocolo estándar de 4 métodos


def list_videos_ffpp(root: str, compression: str = "c23",
                     methods: Sequence[str] = FFPP_METHODS) -> List[VideoItem]:
    """Lee la estructura original de FF++: original_sequences/youtube/<comp>/videos y manipulated_sequences/<metodo>/<comp>/videos."""
    items: List[VideoItem] = []
    d = os.path.join(root, "original_sequences", "youtube", compression, "videos")
    if os.path.isdir(d):
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VIDEO_EXT):
                stem = os.path.splitext(f)[0]
                items.append(VideoItem(os.path.join(d, f), 0, "real", stem, (stem,)))
    for m in methods:
        d = os.path.join(root, "manipulated_sequences", m, compression, "videos")
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VIDEO_EXT):
                stem = os.path.splitext(f)[0]
                t, s_ = stem.split("_")[:2]
                items.append(VideoItem(os.path.join(d, f), 1, m, f"{m}_{stem}", (t, s_)))
    if not items:
        raise FileNotFoundError(f"No hay videos FF++ {compression} en {root}")
    return items


def list_videos_celebdf(root: str, list_file: str = "List_of_testing_videos.txt") -> List[VideoItem]:
    """Lista oficial de test de Celeb-DF v2 (518 videos)."""
    items: List[VideoItem] = []
    with open(os.path.join(root, list_file)) as f:
        for line in f:
            if not line.strip():
                continue
            flag, rel = line.split()
            label = 0 if flag == "1" else 1
            folder, fname = rel.split("/")
            assert (label == 1) == ("synthesis" in folder.lower()), f"Etiqueta inconsistente: {line}"
            stem = os.path.splitext(fname)[0]
            parts = stem.split("_")
            if folder.lower().startswith("youtube"):
                ids = (f"cdf_yt{stem}",)
            elif label == 1:
                ids = (f"cdf_{parts[0]}", f"cdf_{parts[1]}")
            else:
                ids = (f"cdf_{parts[0]}",)
            items.append(VideoItem(os.path.join(root, folder, fname), label,
                                   "CelebDF" if label else "real", f"cdf_{folder}_{stem}", ids))
    missing = [v.path for v in items if not os.path.exists(v.path)]
    if missing:
        raise FileNotFoundError(f"{len(missing)} videos de la lista de Celeb-DF no existen, p. ej. {missing[0]}")
    return items


def detect_layout(root: str) -> str:
    """'ffpp' (estructura oficial) o 'simple' (carpetas real/ y fake/)."""
    if os.path.isdir(os.path.join(root, "original_sequences")) and \
            os.path.isdir(os.path.join(root, "manipulated_sequences")):
        return "ffpp"
    if os.path.isdir(os.path.join(root, "real")) and os.path.isdir(os.path.join(root, "fake")):
        return "simple"
    raise FileNotFoundError(f"{root}: no tiene real/+fake/ ni original_sequences/+manipulated_sequences/")


def load_videos(root: str, compression: str = "c23") -> List[VideoItem]:
    return list_videos_ffpp(root, compression) if detect_layout(root) == "ffpp" else list_videos(root)


def split_official(videos: List[VideoItem]) -> Dict[str, List[VideoItem]]:
    """Partición oficial de FF++ (720/140/140 por video original)."""
    from ff_splits import official_split_of_id

    sp = official_split_of_id()
    for v in videos:
        s = {sp[i] for i in v.identities if i in sp}
        if len(s) != 1:
            raise ValueError(f"{v.video_id}: identidades {v.identities} fuera de la partición oficial o en "
                             f"conjuntos distintos ({s})")
        v.split = s.pop()
    assign_identity_groups(videos)
    return {n: [v for v in videos if v.split == n] for n in ("train", "val", "test")}


def find_dataset_dir(search_roots: Sequence[str]) -> Optional[str]:
    """Busca la primera carpeta con real/+fake/ o con la estructura oficial de FF++."""
    for root in search_roots:
        if not os.path.isdir(root):
            continue
        for cur, dirs, _ in os.walk(root):
            if ("real" in dirs and "fake" in dirs) or \
                    ("original_sequences" in dirs and "manipulated_sequences" in dirs):
                return cur
            if cur.count(os.sep) - root.count(os.sep) >= 4:
                dirs[:] = []
    return None


def quick_subset(videos: List[VideoItem], n_real: int = 25) -> List[VideoItem]:
    """n_real videos reales y un fake por cada uno, rotando el método. Solo para pruebas rápidas."""
    reals = sorted([v for v in videos if v.label == 0], key=lambda v: v.video_id)[:n_real]
    targets = [v.video_id for v in reals]
    by_target: Dict[str, List[VideoItem]] = defaultdict(list)
    for v in videos:
        if v.label == 1:
            by_target[v.identities[0]].append(v)
    fakes = []
    for i, t in enumerate(targets):
        cands = sorted(by_target.get(t, []), key=lambda v: v.method)
        if cands:
            fakes.append(cands[i % len(cands)])
    return reals + fakes


def smoke_subset(videos: List[VideoItem], n_per_class: int = 3) -> List[VideoItem]:
    """n reales y n fakes con identidades distintas; con n=3 queda 1 real y 1 fake por conjunto."""
    fakes, used = [], set()
    for v in sorted((v for v in videos if v.label == 1), key=lambda v: (v.identities[0], v.method)):
        if not used & set(v.identities) and v.method not in {f.method for f in fakes}:
            fakes.append(v)
            used |= set(v.identities)
        if len(fakes) == n_per_class:
            break
    fake_ids = {i for v in videos if v.label == 1 for i in v.identities}  # evita reales ligados a cualquier fake
    reals = [v for v in sorted((v for v in videos if v.label == 0), key=lambda v: v.video_id)
             if not set(v.identities) & fake_ids][:n_per_class]
    if len(fakes) < n_per_class or len(reals) < n_per_class:
        raise ValueError("No hay suficientes videos con identidades disjuntas para el smoke test")
    return reals + fakes


# Partición por identidad

class _UnionFind:
    def __init__(self) -> None:
        self.p: Dict[str, str] = {}

    def find(self, a: str) -> str:
        self.p.setdefault(a, a)
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]
            a = self.p[a]
        return a

    def union(self, a: str, b: str) -> None:
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.p[rb] = ra


def assign_identity_groups(videos: List[VideoItem]) -> int:
    uf = _UnionFind()
    for v in videos:
        for i in v.identities:
            uf.find(i)
        for a, b in zip(v.identities, v.identities[1:]):
            uf.union(a, b)
    roots = sorted({uf.find(v.identities[0]) for v in videos})
    rid = {r: k for k, r in enumerate(roots)}
    for v in videos:
        v.group = rid[uf.find(v.identities[0])]
    return len(roots)


def split_by_identity(videos: List[VideoItem], ratios: Tuple[float, float, float] = (0.70, 0.15, 0.15),
                      seed: int = 42) -> Dict[str, List[VideoItem]]:
    """Reparte grupos de identidad completos entre train, val y test, cuidando que haya reales y fakes en cada uno."""
    assert abs(sum(ratios) - 1.0) < 1e-6
    assign_identity_groups(videos)
    groups: Dict[int, List[VideoItem]] = defaultdict(list)
    for v in videos:
        groups[v.group].append(v)
    strata: Dict[Tuple[bool, bool], List[int]] = defaultdict(list)
    for g, vs in groups.items():
        strata[(any(v.label == 0 for v in vs), any(v.label == 1 for v in vs))].append(g)
    names = ("train", "val", "test")
    rng = np.random.RandomState(seed)
    for key in sorted(strata):
        gids = sorted(strata[key])
        rng.shuffle(gids)
        total = sum(len(groups[g]) for g in gids)
        target = np.array(ratios) * total
        count = np.zeros(3)
        for g in gids:
            k = int(np.argmax((target - count) / np.maximum(target, 1e-9)))
            count[k] += len(groups[g])
            for v in groups[g]:
                v.split = names[k]
    return {n: [v for v in videos if v.split == n] for n in names}


def check_identity_leakage(splits: Dict[str, List[VideoItem]]) -> Dict[str, int]:
    """Cuenta identidades y videos compartidos entre conjuntos."""
    ids = {k: {i for v in vs for i in v.identities} for k, vs in splits.items()}
    paths = {k: {v.path for v in vs} for k, vs in splits.items()}
    out = {}
    for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
        out[f"identidades {a}-{b}"] = len(ids[a] & ids[b])
        out[f"videos {a}-{b}"] = len(paths[a] & paths[b])
    return out


# Alineación facial (MediaPipe)

_TASK_URL = ("https://storage.googleapis.com/mediapipe-models/face_landmarker/"
             "face_landmarker/float16/1/face_landmarker.task")
_LE_IN, _LE_OUT, _RE_IN, _RE_OUT = 133, 33, 362, 263
ALIGN_VERSION = 3
_NOSE_TIP, _CHEEK_L, _CHEEK_R = 1, 234, 454


def yaw_index(pts: np.ndarray) -> float:
    """Giro de la cabeza en [-1, 1]: posición de la nariz respecto al centro de las mejillas (0 = frontal)."""
    xl, xr, xn = float(pts[_CHEEK_L, 0]), float(pts[_CHEEK_R, 0]), float(pts[_NOSE_TIP, 0])
    half = abs(xr - xl) / 2.0
    return float(np.clip((xn - (xl + xr) / 2.0) / max(half, 1e-6), -1.0, 1.0))


class FaceAligner:
    """Alinea el rostro como preprocess.py: nivela los ojos y recorta un cuadrado centrado de 224x224."""

    def __init__(self, output_size: int = 224, margin_ratio: float = 1.35,
                 task_model_path: str = "face_landmarker.task", small_face_fallback: bool = True) -> None:
        import mediapipe as mp

        self.output_size, self.margin = output_size, margin_ratio
        self.small_face_fallback = small_face_fallback
        self._haar = None
        if small_face_fallback:
            try:
                xml = os.path.join(cv2.data.haarcascades, "haarcascade_frontalface_default.xml")
                self._haar = cv2.CascadeClassifier(xml)
                if self._haar.empty():
                    self._haar = None
            except Exception:  # build de OpenCV sin cv2.data
                self._haar = None
        if hasattr(mp, "solutions"):
            self.kind = "solutions"
            self._det = mp.solutions.face_mesh.FaceMesh(
                static_image_mode=True, max_num_faces=1, refine_landmarks=True,
                min_detection_confidence=0.5)
        else:
            from mediapipe.tasks.python import BaseOptions, vision

            if not os.path.exists(task_model_path):
                urllib.request.urlretrieve(_TASK_URL, task_model_path)
            self.kind = "tasks"
            self._mp = mp
            opts = vision.FaceLandmarkerOptions(
                base_options=BaseOptions(model_asset_path=task_model_path),
                running_mode=vision.RunningMode.IMAGE, num_faces=1,
                min_face_detection_confidence=0.5)
            self._det = vision.FaceLandmarker.create_from_options(opts)

    def landmarks(self, frame_rgb: np.ndarray) -> Optional[np.ndarray]:
        """Devuelve (468+, 2) en píxeles o None."""
        pts = self._landmarks_raw(frame_rgb)
        if pts is not None or not self.small_face_fallback:
            return pts
        return self._landmarks_small_face(frame_rgb)

    def _landmarks_small_face(self, frame_rgb: np.ndarray) -> Optional[np.ndarray]:
        if self._haar is None:
            return None
        H, W = frame_rgb.shape[:2]
        gray = cv2.equalizeHist(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2GRAY))
        boxes = self._haar.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(24, 24))
        for (x, y, w, h) in sorted(boxes, key=lambda b: -b[2] * b[3])[:3]:  # los 3 más grandes
            cx, cy, half = x + w / 2.0, y + h / 2.0, 1.1 * max(w, h)
            xa, ya = int(max(0, cx - half)), int(max(0, cy - half))
            xb, yb = int(min(W, cx + half)), int(min(H, cy + half))
            crop = frame_rgb[ya:yb, xa:xb]
            if crop.size == 0:
                continue
            s = 480.0 / max(crop.shape[:2])
            big = cv2.resize(crop, None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC)
            pts = self._landmarks_raw(big)
            if pts is not None:
                return pts / s + np.array([xa, ya], dtype=np.float32)
        return None

    def _landmarks_raw(self, frame_rgb: np.ndarray) -> Optional[np.ndarray]:
        H, W = frame_rgb.shape[:2]
        if self.kind == "solutions":
            res = self._det.process(frame_rgb)
            if not res.multi_face_landmarks:
                return None
            lms = res.multi_face_landmarks[0].landmark
        else:
            img = self._mp.Image(image_format=self._mp.ImageFormat.SRGB, data=np.ascontiguousarray(frame_rgb))
            res = self._det.detect(img)
            if not res.face_landmarks:
                return None
            lms = res.face_landmarks[0]
        return np.array([[l.x * W, l.y * H] for l in lms], dtype=np.float32)

    def align(self, frame_bgr: np.ndarray) -> Tuple[Optional[np.ndarray], Dict]:
        H, W = frame_bgr.shape[:2]
        rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        pts = self.landmarks(rgb)
        if pts is None:
            return None, {}
        lc = (pts[_LE_IN] + pts[_LE_OUT]) / 2.0
        rc = (pts[_RE_IN] + pts[_RE_OUT]) / 2.0
        eye_c = (lc + rc) / 2.0
        angle = float(np.degrees(np.arctan2(rc[1] - lc[1], rc[0] - lc[0])))
        M = cv2.getRotationMatrix2D((float(eye_c[0]), float(eye_c[1])), angle, 1.0)
        aligned = cv2.warpAffine(rgb, M, (W, H), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT_101)
        rp = (M[:, :2] @ pts.T + M[:, 2:]).T
        nl = (rp[_LE_IN] + rp[_LE_OUT]) / 2.0
        nr = (rp[_RE_IN] + rp[_RE_OUT]) / 2.0
        y0, y1 = float(rp[:, 1].min()), float(rp[:, 1].max())
        x0, x1 = float(rp[:, 0].min()), float(rp[:, 0].max())
        cx, cy = float((nl[0] + nr[0]) / 2.0), (y0 + y1) / 2.0
        size = int(round(max((y1 - y0) * self.margin, (x1 - x0) * self.margin,
                             float(np.linalg.norm(nr - nl)) * 2.8)))
        # Sin tamaño mínimo: el encuadre es relativo al rostro (misma escala en todos los videos).
        # (preprocess.py forzaba al menos 224 px y en planos abiertos el rostro quedaba pequeño)
        size = max(size, 16)
        xa, ya = int(round(cx - size / 2.0)), int(round(cy - size / 2.0))
        xb, yb = xa + size, ya + size
        pt, pb, pl, pr = max(0, -ya), max(0, yb - H), max(0, -xa), max(0, xb - W)
        if pt or pb or pl or pr:
            aligned = cv2.copyMakeBorder(aligned, pt, pb, pl, pr, borderType=cv2.BORDER_REFLECT_101)
        crop = aligned[ya + pt:yb + pt, xa + pl:xb + pl]
        if crop.size == 0:
            return None, {}
        face = cv2.resize(crop, (self.output_size, self.output_size), interpolation=cv2.INTER_LINEAR)
        return face, {"angle": angle, "landmarks": pts, "crop": (xa, ya, xb, yb), "yaw": yaw_index(rp)}

    def close(self) -> None:
        try:
            self._det.close()
        except Exception:
            pass


def sample_frame_indices(total: int, n: int) -> List[int]:
    """Muestreo uniforme de N índices (igual que dataset.py, sin duplicar frames)."""
    if total <= 0:
        return []
    if total <= n:
        return list(range(total))
    step = total / n
    return [int(i * step) for i in range(n)]


def read_frames(path: str, indices: Sequence[int]) -> List[Tuple[int, np.ndarray]]:
    wanted, out = set(indices), []
    cap = cv2.VideoCapture(path)
    fi = 0
    last = max(indices) if indices else -1
    while fi <= last:
        ok, frame = cap.read()
        if not ok:
            break
        if fi in wanted:
            out.append((fi, frame))
        fi += 1
    cap.release()
    return out


def build_face_cache(videos: List[VideoItem], n_frames: int, aligner: FaceAligner,
                     cache_file: str, verbose: bool = True) -> Dict[str, np.ndarray]:
    """Extrae y alinea N frames por video."""
    paths = np.array([v.path for v in videos])
    if os.path.exists(cache_file):
        c = dict(np.load(cache_file, allow_pickle=False))
        if (c["paths"].shape == paths.shape and (c["paths"] == paths).all() and int(c["n_frames"]) == n_frames
                and int(c.get("align_version", 0)) == ALIGN_VERSION):
            if verbose:
                print(f"Caché reutilizada: {cache_file}")
            return c
    faces, fvid, fidx, fyaw = [], [], [], []
    n_detect = np.zeros(len(videos), dtype=np.int32)
    t0 = time.time()
    for vi, v in enumerate(videos):
        cap = cv2.VideoCapture(v.path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        for fi, frame in read_frames(v.path, sample_frame_indices(total, n_frames)):
            face, info = aligner.align(frame)
            if face is not None:
                faces.append(face)
                fyaw.append(info["yaw"])
                fvid.append(vi)
                fidx.append(fi)
                n_detect[vi] += 1
        if verbose and ((vi + 1) % 10 == 0 or vi + 1 == len(videos)):
            print(f"  {vi + 1:4d}/{len(videos)} videos | {len(faces)} rostros | {time.time() - t0:.0f}s")
    c = {
        "faces": np.stack(faces).astype(np.uint8),
        "frame_video": np.array(fvid, dtype=np.int32),
        "frame_index": np.array(fidx, dtype=np.int32),
        "frame_yaw": np.array(fyaw, dtype=np.float32),
        "n_detect": n_detect,
        "labels": np.array([v.label for v in videos], dtype=np.int64),
        "paths": paths,
        "n_frames": np.array(n_frames),
        "align_version": np.array(ALIGN_VERSION),
    }
    os.makedirs(os.path.dirname(cache_file) or ".", exist_ok=True)
    np.savez(cache_file, **c)
    return c


# Caché en carpeta (FF++ completo, memoria mapeada)
# Con FF++ completo los rostros ocupan varios GB: se guardan en faces.npy y se leen con
# se reparte entre procesos (MediaPipe corre en CPU; Kaggle ofrece 4 núcleos).

_WORKER_ALIGNER: Optional[FaceAligner] = None


def _worker_init(output_size: int) -> None:
    global _WORKER_ALIGNER
    _WORKER_ALIGNER = FaceAligner(output_size)


def _worker_video(args: Tuple[str, int, int]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Un video -> (rostros (k,224,224,3) uint8, índices de frame, yaw)."""
    path, n_grid, budget = args
    cap = cv2.VideoCapture(path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    grid = sample_frame_indices(total, n_grid)
    if budget < len(grid):
        grid = [grid[i] for i in np.linspace(0, len(grid) - 1, budget).round().astype(int)]
    faces, idx, yaw = [], [], []
    for fi, frame in read_frames(path, grid):
        face, info = _WORKER_ALIGNER.align(frame)
        if face is not None:
            faces.append(face)
            idx.append(fi)
            yaw.append(info["yaw"])
    sz = _WORKER_ALIGNER.output_size
    return (np.stack(faces) if faces else np.zeros((0, sz, sz, 3), np.uint8),
            np.array(idx, np.int32), np.array(yaw, np.float32))


def build_face_cache_dir(videos: List[VideoItem], cache_dir: str, n_grid: int = 16,
                         budgets: Optional[Sequence[int]] = None, workers: int = 1,
                         output_size: int = 224, verbose: bool = True) -> Dict[str, np.ndarray]:
    """Extrae rostros alineados a cache_dir/{faces.npy, meta.npz}."""
    budgets = np.array(budgets if budgets is not None else [n_grid] * len(videos), dtype=np.int32)
    paths = np.array([v.path for v in videos])
    meta_f, faces_f = os.path.join(cache_dir, "meta.npz"), os.path.join(cache_dir, "faces.npy")
    if os.path.exists(meta_f) and os.path.exists(faces_f):
        old = load_face_cache_dir(cache_dir)
        if (old["paths"].shape == paths.shape and (old["paths"] == paths).all()
                and (old["budgets"] == budgets).all() and int(old["n_grid"]) == n_grid
                and int(old["align_version"]) == ALIGN_VERSION):
            if verbose:
                print(f"Caché reutilizada: {cache_dir} ({len(old['frame_video'])} rostros)")
            return old
    os.makedirs(cache_dir, exist_ok=True)
    tmp_f = faces_f + ".tmp.npy"
    big = np.lib.format.open_memmap(tmp_f, mode="w+", dtype=np.uint8,
                                    shape=(int(budgets.sum()), output_size, output_size, 3))
    fvid, fidx, fyaw = [], [], []
    n_detect = np.zeros(len(videos), np.int32)
    jobs = [(v.path, n_grid, int(b)) for v, b in zip(videos, budgets)]
    t0, m = time.time(), 0
    if workers > 1:
        import multiprocessing as mp
        FaceAligner(output_size).close()  # descarga el modelo .task una sola vez
        # "spawn": procesos limpios. MediaPipe/TFLite no es seguro tras fork (corrompe el heap).
        pool = mp.get_context("spawn").Pool(workers, initializer=_worker_init, initargs=(output_size,))
        it = pool.imap(_worker_video, jobs, chunksize=4)
    else:
        _worker_init(output_size)
        pool, it = None, map(_worker_video, jobs)
    for vi, (f, i, y) in enumerate(it):
        k = len(f)
        big[m:m + k] = f
        fvid += [vi] * k
        fidx += list(i)
        fyaw += list(y)
        n_detect[vi] = k
        m += k
        if verbose and ((vi + 1) % 100 == 0 or vi + 1 == len(videos)):
            print(f"  {vi + 1:5d}/{len(videos)} videos | {m} rostros | {time.time() - t0:.0f}s")
    if pool is not None:
        pool.close()
        pool.join()
    big.flush()
    out = np.lib.format.open_memmap(faces_f, mode="w+", dtype=np.uint8, shape=(m, output_size, output_size, 3))
    for s in range(0, m, 2048):
        out[s:s + 2048] = big[s:s + 2048]
    out.flush()
    del big, out
    os.remove(tmp_f)
    np.savez(meta_f, frame_video=np.array(fvid, np.int32), frame_index=np.array(fidx, np.int32),
             frame_yaw=np.array(fyaw, np.float32), n_detect=n_detect,
             labels=np.array([v.label for v in videos], np.int64), paths=paths, budgets=budgets,
             n_grid=np.array(n_grid), align_version=np.array(ALIGN_VERSION),
             video_ids=np.array([v.video_id for v in videos]))
    return load_face_cache_dir(cache_dir)


def load_face_cache_dir(cache_dir: str) -> Dict[str, np.ndarray]:
    c = dict(np.load(os.path.join(cache_dir, "meta.npz"), allow_pickle=False))
    c["faces"] = np.load(os.path.join(cache_dir, "faces.npy"), mmap_mode="r")
    assert len(c["faces"]) == len(c["frame_video"]), "faces.npy y meta.npz no coinciden"
    return c


In [ ]:
%%writefile code_kaggle/ff_splits.py
"""Partición oficial de FaceForensics++ (720/140/140).

Fuente: https://github.com/ondyari/FaceForensics/tree/master/dataset/splits
"""
from typing import Dict, List

OFFICIAL_PAIRS: Dict[str, List[List[str]]] = {
    "train": [["071", "054"], ["087", "081"], ["881", "856"], ["187", "234"], ["645", "688"], ["754", "758"], ["811", "920"], ["710", "788"], ["628", "568"], ["312", "021"], ["950", "836"], ["059", "050"], ["524", "580"], ["751", "752"], ["918", "934"], ["604", "703"], ["296", "293"], ["518", "131"], ["536", "540"], ["969", "897"], ["372", "413"], ["357", "432"], ["809", "799"], ["092", "098"], ["302", "323"], ["981", "985"], ["512", "495"], ["088", "060"], ["795", "907"], ["535", "587"], ["297", "270"], ["838", "810"], ["850", "764"], ["476", "400"], ["268", "269"], ["033", "097"], ["226", "491"], ["784", "769"], ["195", "442"], ["678", "460"], ["320", "328"], ["451", "449"], ["409", "382"], ["556", "588"], ["027", "009"], ["196", "310"], ["241", "210"], ["295", "099"], ["043", "110"], ["753", "789"], ["716", "712"], ["508", "831"], ["005", "010"], ["276", "185"], ["498", "433"], ["294", "292"], ["105", "180"], ["984", "967"], ["318", "334"], ["356", "324"], ["344", "020"], ["289", "228"], ["022", "489"], ["137", "165"], ["095", "053"], ["999", "960"], ["481", "469"], ["534", "490"], ["543", "559"], ["150", "153"], ["598", "178"], ["475", "265"], ["671", "677"], ["204", "230"], ["863", "853"], ["561", "998"], ["163", "031"], ["655", "444"], ["038", "125"], ["735", "774"], ["184", "205"], ["499", "539"], ["717", "684"], ["878", "866"], ["127", "129"], ["286", "267"], ["032", "944"], ["681", "711"], ["236", "237"], ["989", "993"], ["537", "563"], ["814", "871"], ["509", "525"], ["221", "206"], ["808", "829"], ["696", "686"], ["431", "447"], ["737", "719"], ["609", "596"], ["408", "424"], ["976", "954"], ["156", "243"], ["434", "438"], ["627", "658"], ["025", "067"], ["635", "642"], ["523", "541"], ["572", "554"], ["215", "208"], ["651", "835"], ["975", "978"], ["792", "903"], ["931", "936"], ["846", "845"], ["899", "914"], ["209", "016"], ["398", "457"], ["797", "844"], ["360", "437"], ["738", "804"], ["694", "767"], ["790", "014"], ["657", "644"], ["374", "407"], ["728", "673"], ["193", "030"], ["876", "891"], ["553", "545"], ["331", "260"], ["873", "872"], ["109", "107"], ["121", "093"], ["143", "140"], ["778", "798"], ["983", "113"], ["504", "502"], ["709", "390"], ["940", "941"], ["894", "848"], ["311", "387"], ["562", "626"], ["330", "162"], ["112", "892"], ["765", "867"], ["124", "085"], ["665", "679"], ["414", "385"], ["555", "516"], ["072", "037"], ["086", "090"], ["202", "348"], ["341", "340"], ["333", "377"], ["082", "103"], ["569", "921"], ["750", "743"], ["211", "177"], ["770", "791"], ["329", "327"], ["613", "685"], ["007", "132"], ["304", "300"], ["860", "905"], ["986", "994"], ["378", "368"], ["761", "766"], ["232", "248"], ["136", "285"], ["601", "653"], ["693", "698"], ["359", "317"], ["246", "258"], ["500", "592"], ["776", "676"], ["262", "301"], ["307", "365"], ["600", "505"], ["833", "826"], ["361", "448"], ["473", "366"], ["885", "802"], ["277", "335"], ["667", "446"], ["522", "337"], ["018", "019"], ["430", "459"], ["886", "877"], ["456", "435"], ["239", "218"], ["771", "849"], ["065", "089"], ["654", "648"], ["151", "225"], ["152", "149"], ["229", "247"], ["624", "570"], ["290", "240"], ["011", "805"], ["461", "250"], ["251", "375"], ["639", "841"], ["602", "397"], ["028", "068"], ["338", "336"], ["964", "174"], ["782", "787"], ["478", "506"], ["313", "283"], ["659", "749"], ["690", "689"], ["893", "913"], ["197", "224"], ["253", "183"], ["373", "394"], ["803", "017"], ["305", "513"], ["051", "332"], ["238", "282"], ["621", "546"], ["401", "395"], ["510", "528"], ["410", "411"], ["049", "946"], ["663", "231"], ["477", "487"], ["252", "266"], ["952", "882"], ["315", "322"], ["216", "164"], ["061", "080"], ["603", "575"], ["828", "830"], ["723", "704"], ["870", "001"], ["201", "203"], ["652", "773"], ["108", "052"], ["272", "396"], ["040", "997"], ["988", "966"], ["281", "474"], ["077", "100"], ["146", "256"], ["972", "718"], ["303", "309"], ["582", "172"], ["222", "168"], ["884", "968"], ["217", "117"], ["118", "120"], ["242", "182"], ["858", "861"], ["101", "096"], ["697", "581"], ["763", "930"], ["839", "864"], ["542", "520"], ["122", "144"], ["687", "615"], ["544", "532"], ["721", "715"], ["179", "212"], ["591", "605"], ["275", "887"], ["996", "056"], ["825", "074"], ["530", "594"], ["757", "573"], ["611", "760"], ["189", "200"], ["392", "339"], ["734", "699"], ["977", "075"], ["879", "963"], ["910", "911"], ["889", "045"], ["962", "929"], ["515", "519"], ["062", "066"], ["937", "888"], ["199", "181"], ["785", "736"], ["079", "076"], ["155", "576"], ["748", "355"], ["819", "786"], ["577", "593"], ["464", "463"], ["439", "441"], ["574", "547"], ["747", "854"], ["403", "497"], ["965", "948"], ["726", "713"], ["943", "942"], ["160", "928"], ["496", "417"], ["700", "813"], ["756", "503"], ["213", "083"], ["039", "058"], ["781", "806"], ["620", "619"], ["351", "346"], ["959", "957"], ["264", "271"], ["006", "002"], ["391", "406"], ["631", "551"], ["501", "326"], ["412", "274"], ["641", "662"], ["111", "094"], ["166", "167"], ["130", "139"], ["938", "987"], ["055", "147"], ["990", "008"], ["013", "883"], ["614", "616"], ["772", "708"], ["840", "800"], ["415", "484"], ["287", "426"], ["680", "486"], ["057", "070"], ["590", "034"], ["194", "235"], ["291", "874"], ["902", "901"], ["343", "363"], ["279", "298"], ["393", "405"], ["674", "744"], ["244", "822"], ["133", "148"], ["636", "578"], ["637", "427"], ["041", "063"], ["869", "780"], ["733", "935"], ["259", "345"], ["069", "961"], ["783", "916"], ["191", "188"], ["526", "436"], ["123", "119"], ["207", "908"], ["796", "740"], ["815", "730"], ["173", "171"], ["383", "353"], ["458", "722"], ["533", "450"], ["618", "629"], ["646", "643"], ["531", "549"], ["428", "466"], ["859", "843"], ["692", "610"]],
    "val": [["720", "672"], ["939", "115"], ["284", "263"], ["402", "453"], ["820", "818"], ["762", "832"], ["834", "852"], ["922", "898"], ["104", "126"], ["106", "198"], ["159", "175"], ["416", "342"], ["857", "909"], ["599", "585"], ["443", "514"], ["566", "617"], ["472", "511"], ["325", "492"], ["816", "649"], ["583", "558"], ["933", "925"], ["419", "824"], ["465", "482"], ["565", "589"], ["261", "254"], ["992", "980"], ["157", "245"], ["571", "746"], ["947", "951"], ["926", "900"], ["493", "538"], ["468", "470"], ["915", "895"], ["362", "354"], ["440", "364"], ["640", "638"], ["827", "817"], ["793", "768"], ["837", "890"], ["004", "982"], ["192", "134"], ["745", "777"], ["299", "145"], ["742", "775"], ["586", "223"], ["483", "370"], ["779", "794"], ["971", "564"], ["273", "807"], ["991", "064"], ["664", "668"], ["823", "584"], ["656", "666"], ["557", "560"], ["471", "455"], ["042", "084"], ["979", "875"], ["316", "369"], ["091", "116"], ["023", "923"], ["702", "612"], ["904", "046"], ["647", "622"], ["958", "956"], ["606", "567"], ["632", "548"], ["927", "912"], ["350", "349"], ["595", "597"], ["727", "729"]],
    "test": [["953", "974"], ["012", "026"], ["078", "955"], ["623", "630"], ["919", "015"], ["367", "371"], ["847", "906"], ["529", "633"], ["418", "507"], ["227", "169"], ["389", "480"], ["821", "812"], ["670", "661"], ["158", "379"], ["423", "421"], ["352", "319"], ["579", "701"], ["488", "399"], ["695", "422"], ["288", "321"], ["705", "707"], ["306", "278"], ["865", "739"], ["995", "233"], ["755", "759"], ["467", "462"], ["314", "347"], ["741", "731"], ["970", "973"], ["634", "660"], ["494", "445"], ["706", "479"], ["186", "170"], ["176", "190"], ["380", "358"], ["214", "255"], ["454", "527"], ["425", "485"], ["388", "308"], ["384", "932"], ["035", "036"], ["257", "420"], ["924", "917"], ["114", "102"], ["732", "691"], ["550", "452"], ["280", "249"], ["842", "714"], ["625", "650"], ["024", "073"], ["044", "945"], ["896", "128"], ["862", "047"], ["607", "683"], ["517", "521"], ["682", "669"], ["138", "142"], ["552", "851"], ["376", "381"], ["000", "003"], ["048", "029"], ["724", "725"], ["608", "675"], ["386", "154"], ["220", "219"], ["801", "855"], ["161", "141"], ["949", "868"], ["880", "135"], ["429", "404"]],
}


def official_split_of_id() -> Dict[str, str]:
    """id de video original ("000"..."999") -> "train" | "val" | "test"."""
    out: Dict[str, str] = {}
    for split, pairs in OFFICIAL_PAIRS.items():
        for a, b in pairs:
            out[a] = out[b] = split
    return out


In [ ]:
%%writefile code_kaggle/metrics.py
"""Métricas (AUC, F1, precisión, recall, EER), intervalos bootstrap y figuras."""
from __future__ import annotations

import os
from typing import Dict, List, Optional, Sequence

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_recall_fscore_support,
                             roc_auc_score, roc_curve)


def save_fig(fig: matplotlib.figure.Figure, path: str) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    return path


def video_scores(frame_probs: np.ndarray, frame_video: np.ndarray, video_ids: np.ndarray) -> np.ndarray:
    """Agregación temporal (Etapa 3): P_video = promedio de p_i sobre sus frames."""
    out = np.empty(len(video_ids))
    for k, v in enumerate(video_ids):
        out[k] = frame_probs[frame_video == v].mean()
    return out


def youden_threshold(y: np.ndarray, s: np.ndarray) -> float:
    if len(np.unique(y)) < 2:
        return 0.5
    fpr, tpr, thr = roc_curve(y, s)
    return float(np.clip(thr[int(np.argmax(tpr - fpr))], 0.0, 1.0))


def eer(y: np.ndarray, s: np.ndarray) -> float:
    if len(np.unique(y)) < 2:
        return float("nan")
    fpr, tpr, _ = roc_curve(y, s)
    fnr = 1 - tpr
    i = int(np.nanargmin(np.abs(fnr - fpr)))
    return float((fpr[i] + fnr[i]) / 2)


def binary_metrics(y: np.ndarray, s: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y, s = np.asarray(y).astype(int), np.asarray(s, dtype=np.float64)
    pred = (s >= threshold).astype(int)
    auc = roc_auc_score(y, s) if len(np.unique(y)) == 2 else float("nan")
    p, r, f1, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return {"auc": float(auc), "accuracy": float(accuracy_score(y, pred)), "precision": float(p),
            "recall": float(r), "f1": float(f1), "eer": eer(y, s), "threshold": float(threshold),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp), "n": int(len(y))}


# Figuras

def plot_history(histories: Dict[str, List[Dict]], path: str) -> matplotlib.figure.Figure:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for name, h in histories.items():
        e = [r["epoch"] for r in h]
        ax[0].plot(e, [r["train_loss"] for r in h], marker="o", ms=3, label=name)
        ax[1].plot(e, [r["val_auc_buena"] for r in h], marker="o", ms=3, label=f"{name}")
        ax[2].plot(e, [r["val_auc_mala"] for r in h], marker="s", ms=3, label=f"{name}")
    for a, t in zip(ax, ("Focal Loss (train)", "AUC val: calidad buena", "AUC val: calidad mala")):
        a.set_title(t)
        a.set_xlabel("Época")
        a.grid(alpha=0.3)
        a.legend(fontsize=8)
    ax[1].set_ylim(0, 1.02)
    ax[2].set_ylim(0, 1.02)
    fig.tight_layout()
    save_fig(fig, path)
    return fig


def plot_roc(curves: Dict[str, tuple], path: str, title: str) -> matplotlib.figure.Figure:
    fig, ax = plt.subplots(figsize=(5.5, 5))
    for name, (y, s) in curves.items():
        if len(np.unique(y)) < 2:
            continue
        fpr, tpr, _ = roc_curve(y, s)
        ax.plot(fpr, tpr, lw=1.8, label=f"{name} (AUC={roc_auc_score(y, s):.3f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("Tasa de falsos positivos")
    ax.set_ylabel("Tasa de verdaderos positivos")
    ax.set_title(title)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(alpha=0.3)
    save_fig(fig, path)
    return fig


def plot_confusions(cms: Dict[str, np.ndarray], path: str) -> matplotlib.figure.Figure:
    n = len(cms)
    cols = min(n, 4)
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.2 * rows), squeeze=False)
    for a in axes.ravel():
        a.axis("off")
    for a, (name, cm) in zip(axes.ravel(), cms.items()):
        a.axis("on")
        a.imshow(cm, cmap="Blues", vmin=0)
        for i in range(2):
            for j in range(2):
                a.text(j, i, int(cm[i, j]), ha="center", va="center",
                       color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=12)
        a.set_xticks([0, 1], ["Real", "Fake"])
        a.set_yticks([0, 1], ["Real", "Fake"])
        a.set_xlabel("Predicción")
        a.set_ylabel("Etiqueta")
        a.set_title(name, fontsize=9)
    fig.tight_layout()
    save_fig(fig, path)
    return fig


def plot_quality_curve(table: Dict[str, Dict[str, float]], levels: Sequence[str], metric: str,
                       path: str) -> matplotlib.figure.Figure:
    """table[modelo][nivel] = valor de la métrica."""
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.arange(len(levels))
    for name, row in table.items():
        ax.plot(x, [row[l] for l in levels], marker="o", lw=2, label=name)
    ax.set_xticks(x, levels)
    ax.set_xlabel("Calidad de la imagen de prueba")
    ax.set_ylabel(metric.upper())
    ax.set_ylim(0, 1.02)
    ax.axhline(0.5, color="gray", ls=":", lw=1)
    ax.set_title(f"{metric.upper()} a nivel de video según la calidad")
    ax.grid(alpha=0.3)
    ax.legend()
    save_fig(fig, path)
    return fig


# Intervalos de confianza (bootstrap)
# Remuestreo por GRUPOS de identidad (cluster bootstrap): los videos de una misma persona
# Así el intervalo no es artificialmente estrecho.

def _cluster_resample(groups: np.ndarray, rng: np.random.RandomState) -> np.ndarray:
    uniq = np.unique(groups)
    pick = rng.choice(uniq, size=len(uniq), replace=True)
    pos = {g: np.flatnonzero(groups == g) for g in uniq}
    return np.concatenate([pos[g] for g in pick])


def bootstrap_auc(y: np.ndarray, s: np.ndarray, groups: Optional[np.ndarray] = None,
                  n_boot: int = 2000, seed: int = 42, alpha: float = 0.05) -> Dict[str, float]:
    """AUC con IC (1-alpha) por percentiles."""
    y, s = np.asarray(y), np.asarray(s)
    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    rng = np.random.RandomState(seed)
    vals = []
    for _ in range(n_boot):
        i = _cluster_resample(g, rng)
        if len(np.unique(y[i])) == 2:
            vals.append(roc_auc_score(y[i], s[i]))
    vals = np.array(vals)
    return {"auc": float(roc_auc_score(y, s)), "ic_inf": float(np.percentile(vals, 100 * alpha / 2)),
            "ic_sup": float(np.percentile(vals, 100 * (1 - alpha / 2))), "n_boot_validos": int(len(vals))}


def paired_bootstrap_auc_diff(y: np.ndarray, s_a: np.ndarray, s_b: np.ndarray,
                              groups: Optional[np.ndarray] = None, n_boot: int = 2000, seed: int = 42,
                              alpha: float = 0.05) -> Dict[str, float]:
    """Diferencia AUC(a) - AUC(b) sobre los mismos videos, con IC y p-valor bilateral (bootstrap)."""
    y, s_a, s_b = np.asarray(y), np.asarray(s_a), np.asarray(s_b)
    g = np.arange(len(y)) if groups is None else np.asarray(groups)
    rng = np.random.RandomState(seed)
    d = []
    for _ in range(n_boot):
        i = _cluster_resample(g, rng)
        if len(np.unique(y[i])) == 2:
            d.append(roc_auc_score(y[i], s_a[i]) - roc_auc_score(y[i], s_b[i]))
    d = np.array(d)
    obs = float(roc_auc_score(y, s_a) - roc_auc_score(y, s_b))
    p = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
    return {"diferencia": obs, "ic_inf": float(np.percentile(d, 100 * alpha / 2)),
            "ic_sup": float(np.percentile(d, 100 * (1 - alpha / 2))), "p_valor": p}


In [ ]:
%%writefile code_kaggle/trainer.py
"""Entrenamiento y evaluación de un experimento."""
from __future__ import annotations

import json
import os
import time
from dataclasses import asdict, dataclass
from typing import Any, Callable, Dict, Iterator, List, Optional

import numpy as np

import backend as bk
from augment import (QualityAugmenter, RobustNormalizer, apply_degradation, bilateral_diff_map)
from backend import xp
from layers import BatchNorm2d, Module, softmax
from losses import FocalLoss
from metrics import video_scores
from optim import AdamW, CosineLR, clip_grad_norm

Array = Any


@dataclass
class TrainConfig:
    epochs: int = 50
    batch_size: int = 16
    eval_batch_size: int = 32
    lr: float = 1e-4
    weight_decay: float = 1e-4
    clip_norm: float = 5.0
    frames_per_video: int = 8
    patience: int = 7
    focal_gamma: float = 2.0
    focal_alpha: float = 0.75
    select_on: str = "val_auc_buena"
    val_bad_level: str = "mala"
    seed: int = 42
    diff_mode: str = "pixel"
    yaw_mask: float = 0.0  # >0: apaga la rama lateral si |yaw| supera este umbral
    balance_classes: bool = False  # True: igual nº de frames real/fake por época (FF++ es 1:4)


def iter_modules(m: Module) -> Iterator[Module]:
    yield m
    for c in m.children():
        yield from iter_modules(c)


def get_state(model: Module) -> List[np.ndarray]:
    st = [bk.to_numpy(p.data).copy() for p in model.parameters()]
    for m in iter_modules(model):
        if isinstance(m, BatchNorm2d):
            st += [bk.to_numpy(m.run_mean).copy(), bk.to_numpy(m.run_var).copy()]
    return st


def set_state(model: Module, st: List[np.ndarray]) -> None:
    params = model.parameters()
    for p, a in zip(params, st):
        assert p.data.shape == a.shape
        p.data[...] = bk.asarray(a)
    k = len(params)
    for m in iter_modules(model):
        if isinstance(m, BatchNorm2d):
            m.run_mean, m.run_var = bk.asarray(st[k]), bk.asarray(st[k + 1])
            k += 2
    assert k == len(st), "El estado no corresponde a la arquitectura"


class Experiment:
    """Un modelo + sus datos + su régimen de aumento."""

    def __init__(self, name: str, model: Module, cfg: TrainConfig, cache: Dict[str, np.ndarray],
                 split_videos: Dict[str, np.ndarray], face_norm: RobustNormalizer,
                 diff_norm: RobustNormalizer, augmenter: Optional[QualityAugmenter],
                 levels: Dict[str, Dict[str, float]], out_dir: str) -> None:
        self.name, self.model, self.cfg = name, model, cfg
        self.cache = cache
        self.faces, self.fvid = cache["faces"], cache["frame_video"]
        self.labels = cache["labels"]
        self.split_videos = split_videos
        self.face_norm, self.diff_norm, self.aug = face_norm, diff_norm, augmenter
        self.levels, self.out_dir = levels, out_dir
        self.crit = FocalLoss(cfg.focal_gamma, cfg.focal_alpha)
        self.opt = AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
        self.sched = CosineLR(self.opt, cfg.epochs, eta_min=1e-7)
        self.history: List[Dict[str, float]] = []
        self.best_state: Optional[List[np.ndarray]] = None
        self.frames_of = {int(v): np.flatnonzero(self.fvid == v) for v in np.unique(self.fvid)}

    # Preparación del lote
    def prepare(self, idx: np.ndarray, augment: bool, spec: Optional[Dict] = None, seed: int = 0,
                cache: Optional[Dict[str, np.ndarray]] = None):
        """Índices de frame -> (rostro normalizado, mapa de asimetría normalizado, máscara lateral)."""
        c = self.cache if cache is None else cache
        idx = np.sort(idx) if isinstance(c["faces"], np.memmap) else idx  # lectura secuencial del disco
        x = bk.asarray(np.asarray(c["faces"][idx])).transpose(0, 3, 1, 2) / 255.0  # (B,3,224,224) en [0,1]
        if augment and self.aug is not None:
            x = self.aug(x)
        if spec:
            x = apply_degradation(x, spec, seed)
        d = bilateral_diff_map(x, self.cfg.diff_mode)  # (B,3,224,112)
        lmask = None
        if self.cfg.yaw_mask > 0 and "frame_yaw" in c:
            lmask = (np.abs(c["frame_yaw"][idx]) <= self.cfg.yaw_mask).astype(np.float32)
        return self.face_norm(x), self.diff_norm(d), lmask, idx

    # Entrenamiento
    def train_epoch(self, rng: np.random.RandomState) -> float:
        self.model.train()
        tr = [int(v) for v in self.split_videos["train"] if int(v) in self.frames_of]
        k = {0: self.cfg.frames_per_video, 1: self.cfg.frames_per_video}
        if self.cfg.balance_classes:  # mismo nº esperado de frames por clase en cada época
            n_r = sum(self.labels[v] == 0 for v in tr)
            n_f = sum(self.labels[v] == 1 for v in tr)
            k[1] = max(1, int(round(self.cfg.frames_per_video * n_r / max(n_f, 1))))
        idx = []
        for v in tr:
            fr = self.frames_of[v]
            idx.append(rng.choice(fr, size=min(k[int(self.labels[v])], len(fr)), replace=False))
        idx = np.concatenate(idx)
        rng.shuffle(idx)
        bs, losses, n = self.cfg.batch_size, 0.0, 0
        for s in range(0, len(idx), bs):
            b = idx[s:s + bs]
            if len(b) < 2:  # BatchNorm necesita al menos 2 muestras
                continue
            face, diff, lmask, b = self.prepare(b, augment=True)
            y = xp.asarray(self.labels[self.fvid[b]])
            logits = self.model(face, diff, lmask)
            loss, dl = self.crit(logits, y)
            if not np.isfinite(loss):
                raise FloatingPointError(f"Pérdida no finita en {self.name}")
            self.model.backward(dl)
            clip_grad_norm(self.model.parameters(), self.cfg.clip_norm)
            self.opt.step()
            self.model.release()  # libera activaciones antes del siguiente lote
            losses += loss * len(b)
            n += len(b)
        return losses / max(n, 1)

    # Inferencia
    def predict_frames(self, frame_idx: np.ndarray, spec: Optional[Dict] = None, seed: int = 1234,
                       cache: Optional[Dict[str, np.ndarray]] = None) -> np.ndarray:
        """P(fake) por frame, en el mismo orden que frame_idx."""
        self.model.eval()
        out = np.empty(len(frame_idx), np.float64)
        bs = self.cfg.eval_batch_size
        for k, s in enumerate(range(0, len(frame_idx), bs)):
            chunk = frame_idx[s:s + bs]
            face, diff, lmask, used = self.prepare(chunk, augment=False, spec=spec, seed=seed + k, cache=cache)
            p = bk.to_numpy(softmax(self.model(face, diff, lmask), axis=1)[:, 1])
            out[s:s + len(chunk)] = p[np.argsort(np.argsort(chunk))] if used is not chunk else p
            self.model.release()
        return out

    def video_predictions(self, split: str, spec: Optional[Dict] = None):
        vids = self.split_videos[split]
        fidx = np.concatenate([self.frames_of[int(v)] for v in vids])
        p = self.predict_frames(fidx, spec)
        return self.labels[vids], video_scores(p, self.fvid[fidx], vids)

    def predict_on(self, cache: Dict[str, np.ndarray], spec: Optional[Dict] = None):
        """Evalúa sobre otra caché (p."""
        fv = cache["frame_video"]
        vids = np.unique(fv)
        p = self.predict_frames(np.arange(len(fv)), spec, cache=cache)
        return vids, cache["labels"][vids], video_scores(p, fv, vids)

    # Bucle principal
    def fit(self, verbose: bool = True) -> List[Dict[str, float]]:
        from sklearn.metrics import roc_auc_score

        cfg = self.cfg
        rng = np.random.RandomState(cfg.seed)
        best, bad = -np.inf, 0
        for ep in range(1, cfg.epochs + 1):
            t0 = time.time()
            tl = self.train_epoch(rng)
            yb, sb = self.video_predictions("val")
            ym, sm = self.video_predictions("val", self.levels[cfg.val_bad_level])
            auc = lambda y, s: float(roc_auc_score(y, s)) if len(np.unique(y)) == 2 else float("nan")
            rec = {"epoch": ep, "train_loss": tl, "val_auc_buena": auc(yb, sb), "val_auc_mala": auc(ym, sm),
                   "lr": self.opt.lr, "seg": time.time() - t0}
            self.history.append(rec)
            self.sched.step()
            score = rec[cfg.select_on]
            improved = score > best + 1e-6
            if improved:
                best, bad = score, 0
                self.best_state = get_state(self.model)
                self.best_epoch = ep
            else:
                bad += 1
            if verbose:
                print(f"[{self.name}] ép {ep:3d} | loss {tl:.4f} | AUC val buena {rec['val_auc_buena']:.3f}"
                      f" | mala {rec['val_auc_mala']:.3f} | {rec['seg']:.0f}s{'  *' if improved else ''}")
            if bad >= cfg.patience:
                if verbose:
                    print(f"[{self.name}] early stopping en la época {ep}")
                break
        set_state(self.model, self.best_state)
        self.save()
        self.free_training_state()
        return self.history

    def free_training_state(self) -> None:
        """Libera lo que solo sirve para entrenar (estado de AdamW, gradientes, activaciones)."""
        self.opt = self.sched = None
        self.best_state = None
        for p in self.model.parameters():
            p.grad = None
        self.model.release()

    def save(self) -> str:
        d = os.path.join(self.out_dir, "checkpoints")
        os.makedirs(d, exist_ok=True)
        path = os.path.join(d, f"{self.name}_best.npz")
        np.savez(path, *get_state(self.model))
        with open(os.path.join(d, f"{self.name}_meta.json"), "w") as f:
            json.dump({"config": asdict(self.cfg), "best_epoch": getattr(self, "best_epoch", None),
                       "face_norm": self.face_norm.to_dict(), "diff_norm": self.diff_norm.to_dict(),
                       "history": self.history}, f, indent=2)
        return path


In [ ]:
%%writefile code_kaggle/gradcheck.py
"""Verificación numérica de todos los gradientes (diferencias finitas en float64)."""
from __future__ import annotations

import os
import sys
from typing import Any, Callable, List

import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import backend as bk

bk.set_dtype(np.float64)
bk.set_seed(42)
from backend import xp
from cnn import CNNBranch
from detector import DeepfakeDetectorScratch
from layers import (BatchNorm2d, Conv2d, GELU, GlobalAvgPool2d, LayerNorm, Linear, MaxPool2d,
                    Module, Parameter)
from losses import FocalLoss
from mlp import MLPClassifier
from swin import (PatchEmbedSwin, PatchMerging, SwinBlock, SwinBranch, WindowAttention,
                  shifted_window_mask, window_partition, window_reverse)

EPS, TOL, ABS_TOL, N_SAMPLES = 1e-6, 1e-5, 1e-8, 12
_rng = np.random.RandomState(0)


def _rel(a: float, n: float) -> float:
    if abs(a - n) < ABS_TOL:  # por debajo del redondeo de la derivada numérica
        return 0.0
    return abs(a - n) / (abs(a) + abs(n) + 1e-12)


def _check_array(arr: Any, grad: Any, loss_fn: Callable[[], float]) -> float:
    """Compara `grad` con diferencias centrales en entradas aleatorias de `arr`."""
    worst = 0.0
    flat_idx = _rng.choice(arr.size, size=min(N_SAMPLES, arr.size), replace=False)
    for fi in flat_idx:
        idx = np.unravel_index(int(fi), arr.shape)
        old = float(arr[idx])
        arr[idx] = old + EPS
        lp = loss_fn()
        arr[idx] = old - EPS
        lm = loss_fn()
        arr[idx] = old
        worst = max(worst, _rel(float(grad[idx]), (lp - lm) / (2 * EPS)))
    return worst


def check_module(name: str, mod: Module, x: Any) -> bool:
    """L = sum out*R con R fija; verifica dL/dθ de cada parámetro y dL/dx."""
    out = mod(x)
    R = bk.randn(*out.shape)
    loss_fn = lambda: float((mod(x) * R).sum())
    mod(x)
    dx = mod.backward(R)
    grads = [p.grad.copy() for p in mod.parameters()]
    worst = _check_array(x, dx, loss_fn)
    for p, g in zip(mod.parameters(), grads):
        assert g.shape == p.data.shape, f"{name}: grad {g.shape} ≠ param {p.data.shape}"
        worst = max(worst, _check_array(p.data, g, loss_fn))
    ok = worst < TOL
    print(f"  [{'OK ' if ok else 'FALLA'}] {name:<34} out={tuple(out.shape)}  err_rel_max={worst:.2e}")
    return ok


class _MaskedWA(Module):
    """WindowAttention con la máscara real de un mapa 8x8, ventana 4, desplazamiento 2 (nW=4)."""

    def __init__(self) -> None:
        self.wa = WindowAttention(6, 4, 2)
        self.mask = bk.asarray(shifted_window_mask(8, 8, 4, 2))

    def forward(self, x: Any) -> Any:
        return self.wa(x, self.mask)

    def backward(self, dy: Any) -> Any:
        return self.wa.backward(dy)


def _check_swin_structure() -> bool:
    """Partición/reverso inversos exactos y máscara que bloquea regiones no contiguas tras el roll."""
    x = bk.randn(2, 8, 8, 5)
    inv = bool(xp.array_equal(window_reverse(window_partition(x, 4), 4, 2, 8, 8), x))
    m = shifted_window_mask(8, 8, 4, 2)
    blk = SwinBlock(6, (8, 8), 2, window=4, shift=2)
    blk(bk.randn(1, 64, 6))
    a = bk.to_numpy(blk.attn.attn)  # (nW, h, N, N)
    leak = float(a[np.broadcast_to((m != 0)[:, None], a.shape)].max())
    # ventana 0 sin mezclar regiones (todo 0); la última mezcla 4 regiones
    shape_ok = m.shape == (4, 16, 16) and (m[0] == 0).all() and (m[3] != 0).any()
    ok = inv and shape_ok and leak < 1e-30
    print(f"  [{'OK ' if ok else 'FALLA'}] Swin: partición↔reverso exactos, máscara (4,16,16), "
          f"atención máx. entre regiones bloqueadas={leak:.1e}")
    return ok


def main() -> None:
    results: List[bool] = []
    print(f"Backend: {'CuPy (GPU)' if bk.ON_GPU else 'NumPy (CPU)'} | dtype={bk.DTYPE.__name__}")
    print("1) Gradientes capa por capa")
    results += [
        check_module("Linear", Linear(5, 4), bk.randn(3, 7, 5)),
        check_module("GELU", GELU(), bk.randn(4, 6)),
        check_module("LayerNorm", LayerNorm(6), bk.randn(3, 5, 6)),
        check_module("BatchNorm2d (train)", BatchNorm2d(3), bk.randn(4, 3, 5, 4)),
        check_module("Conv2d k3 s1 p1", Conv2d(3, 4, 3, 1, 1, bias=False), bk.randn(2, 3, 6, 5)),
        check_module("Conv2d k3 s2 p1 +bias", Conv2d(2, 3, 3, 2, 1, bias=True), bk.randn(2, 2, 7, 6)),
        check_module("MaxPool2d", MaxPool2d(2), bk.randn(2, 3, 6, 4)),
        check_module("GlobalAvgPool2d", GlobalAvgPool2d(), bk.randn(2, 3, 4, 5)),
        check_module("PatchEmbedSwin (+LN)", PatchEmbedSwin(8, 2, 3, 6), bk.randn(2, 3, 8, 8)),
        check_module("WindowAttention (sin máscara)", WindowAttention(6, 3, 2), bk.randn(4, 9, 6)),
        check_module("WindowAttention (con máscara)", _MaskedWA(), bk.randn(2 * 4, 16, 6)),
        check_module("SwinBlock W-MSA (shift=0)", SwinBlock(6, (8, 8), 2, window=4, shift=0), bk.randn(2, 64, 6)),
        check_module("SwinBlock SW-MSA (shift=2)", SwinBlock(6, (8, 8), 2, window=4, shift=2), bk.randn(2, 64, 6)),
        check_module("PatchMerging", PatchMerging((4, 6), 5), bk.randn(2, 24, 5)),
    ]
    results.append(_check_swin_structure())
    bn = BatchNorm2d(3); bn(bk.randn(4, 3, 5, 4)); bn.eval()
    results.append(check_module("BatchNorm2d (eval)", bn, bk.randn(4, 3, 5, 4)))

    print("2) Gradientes de los tres modelos (configuración reducida)")
    mlp = MLPClassifier(12, 8, 2, dropout=0.0)
    cnn = CNNBranch(3, (4, 6, 8), proj_dim=6)
    # 16x16 -> parches 2 -> 8x8 (ventana 4, con desplazamiento) -> merging -> 4x4 (ventana = mapa)
    vit = SwinBranch(img_size=16, patch_size=2, embed_dim=4, depths=(2, 2), num_heads=(2, 2),
                     window=4, proj_dim=6)
    results += [
        check_module("MLPClassifier", mlp, bk.randn(4, 12)),
        check_module("CNNBranch", cnn, bk.randn(3, 3, 16, 8)),
        check_module("SwinBranch", vit, bk.randn(2, 3, 16, 16)),
    ]

    print("3) Focal Loss (α=0.75, γ=2) respecto a los logits")
    crit = FocalLoss(2.0, 0.75)
    z = bk.randn(6, 2) * 2
    t = xp.asarray([0, 1, 1, 0, 1, 0])
    _, dz = crit(z, t)
    w = _check_array(z, dz, lambda: crit(z, t)[0])
    results.append(w < TOL)
    print(f"  [{'OK ' if w < TOL else 'FALLA'}] FocalLoss{'':<25} err_rel_max={w:.2e}")
    big = xp.asarray([[800.0, -800.0], [-800.0, 800.0]])
    lb, db = crit(big, xp.asarray([0, 0]))
    fin = bool(np.isfinite(lb) and xp.isfinite(db).all())
    results.append(fin)
    print(f"  [{'OK ' if fin else 'FALLA'}] Estabilidad con logits ±800      loss={lb:.4f} (finita)")

    print("4) Detector completo de extremo a extremo (Swin + CNN + fusión + MLP + Focal Loss)")
    det = DeepfakeDetectorScratch(vit, cnn, MLPClassifier(12, 8, 2, dropout=0.0), drop_branch=0.0)
    face, diff = bk.randn(4, 3, 16, 16), bk.randn(4, 3, 16, 8)
    y = xp.asarray([0, 1, 1, 0])
    loss_fn = lambda: crit(det(face, diff), y)[0]
    _, dl = crit(det(face, diff), y)
    det.backward(dl)
    grads = [p.grad.copy() for p in det.parameters()]
    w = max(_check_array(p.data, g, loss_fn) for p, g in zip(det.parameters(), grads))
    results.append(w < TOL)
    print(f"  [{'OK ' if w < TOL else 'FALLA'}] {len(grads)} tensores de parámetros{'':<10} err_rel_max={w:.2e}")

    lm = np.array([1.0, 0.0, 1.0, 0.0], dtype=np.float32)  # frames 2 y 4 con la rama lateral apagada (giro)

    def loss_mask():
        return crit(det(face, diff, lm), y)[0]
    _, dl = crit(det(face, diff, lm), y)
    det.zero_grad()
    det.backward(dl)
    grads = [p.grad.copy() for p in det.parameters()]
    w = max(_check_array(p.data, g, loss_mask) for p, g in zip(det.parameters(), grads))
    results.append(w < TOL)
    print(f"  [{'OK ' if w < TOL else 'FALLA'}] Máscara lateral por giro{'':<21} err_rel_max={w:.2e}")

    for mode in ("solo_swin", "solo_asimetria"):
        det.branches = mode
        _, dl = crit(det(face, diff), y)
        det.zero_grad()
        det.backward(dl)
        grads = [p.grad.copy() for p in det.parameters()]
        w = max(_check_array(p.data, g, loss_fn) for p, g in zip(det.parameters(), grads))
        results.append(w < TOL)
        print(f"  [{'OK ' if w < TOL else 'FALLA'}] Ablación {mode:<16}{'':<12} err_rel_max={w:.2e}")
    det.branches = "ambas"

    print("5) Dimensiones con los tamaños reales del pipeline (float32, B=2, Swin-T completo 2-2-6-2)")
    bk.set_dtype(np.float32)
    bk.set_seed(42)
    B = 2
    vit_r = SwinBranch()
    cnn_r, mlp_r = CNNBranch(), MLPClassifier()
    det_r = DeepfakeDetectorScratch(vit_r, cnn_r, mlp_r)
    face, diff = bk.randn(B, 3, 224, 224), bk.randn(B, 3, 224, 112)
    g768 = vit_r.forward_features(face)
    fg, fl = vit_r(face), cnn_r(diff)
    logits = det_r(face, diff)
    loss, dl = FocalLoss()(logits, xp.asarray([0, 1]))
    det_r.backward(dl)
    shapes = {
        "Swin vector global": (g768.shape, (B, 768)),
        "Swin proyección": (fg.shape, (B, 256)),
        "CNN proyección": (fl.shape, (B, 256)),
        "logits": (logits.shape, (B, 2)),
        "dL/dlogits": (dl.shape, (B, 2)),
    }
    for k, (got, exp) in shapes.items():
        ok = tuple(got) == exp
        results.append(ok)
        print(f"  [{'OK ' if ok else 'FALLA'}] {k:<20} {tuple(got)}")
    ok = all(p.grad.shape == p.data.shape and p.grad.dtype == np.float32 and bool(xp.isfinite(p.grad).all())
             for p in det_r.parameters())
    results.append(ok)
    print(f"  [{'OK ' if ok else 'FALLA'}] grads finitos, float32 y con la forma de su parámetro | loss={loss:.4f}")
    pv = det_r.predict_video(bk.randn(3, 3, 224, 224), bk.randn(3, 3, 224, 112))
    results.append(0.0 <= pv <= 1.0)
    print(f"  [OK ] predict_video → P(deepfake)={pv:.4f}")
    print(f"  Parámetros: CNN={cnn_r.n_params():,} | MLP={mlp_r.n_params():,} | Swin-T={vit_r.n_params():,}")

    print(f"\nRESULTADO: {sum(results)}/{len(results)} verificaciones correctas")
    sys.exit(0 if all(results) else 1)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile code_kaggle/experimento.py
"""Protocolo FF++ en tres fases: preprocesar, entrenar y evaluar, analizar."""
from __future__ import annotations

import glob
import json
import os
import random
import time
from dataclasses import asdict, dataclass, field
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np

import backend as bk
from augment import (DIFF_MODES, ISOLATED_DEGRADATIONS, QUALITY_LEVELS, AugConfig, QualityAugmenter,
                     RobustNormalizer, bilateral_diff_map)
from data import (VideoItem, assign_identity_groups, build_face_cache_dir, check_identity_leakage,
                  detect_layout, list_videos_celebdf, load_face_cache_dir, load_videos, split_by_identity,
                  split_official)

EXPERIMENTS: Dict[str, Tuple[str, bool]] = {  # nombre -> (ramas, ¿aumento de calidad?)
    "robusto_con_aumento": ("ambas", True),  # modelo propuesto
    "base_sin_aumento": ("ambas", False),
    "ablacion_solo_swin": ("solo_swin", True),
    "ablacion_solo_asimetria": ("solo_asimetria", True),
}
SYNTH_TEST = {**QUALITY_LEVELS, "jpeg_q30_no_visto": ISOLATED_DEGRADATIONS["jpeg_q30_no_visto"]}


def set_seeds(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    bk.set_seed(seed)


@dataclass
class PrepConfig:
    dataset_root: str
    out_dir: str
    celebdf_root: Optional[str] = None
    n_grid: int = 16
    budget_train_real: int = 16
    budget_train_fake: int = 4
    budget_val: int = 8
    budget_test: int = 16
    workers: int = 4
    norm_fit_frames: int = 2048
    max_videos_per_split: Optional[int] = None  # solo para pruebas rápidas
    seed: int = 42


def _limit(vs: List[VideoItem], n: Optional[int], seed: int) -> List[VideoItem]:
    if n is None or len(vs) <= n:
        return vs
    rng = np.random.RandomState(seed)
    reals = [v for v in vs if v.label == 0]
    fakes = [v for v in vs if v.label == 1]
    k = n // 2
    pick = [reals[i] for i in sorted(rng.choice(len(reals), min(k, len(reals)), replace=False))] + \
           [fakes[i] for i in sorted(rng.choice(len(fakes), min(n - k, len(fakes)), replace=False))]
    return pick


def preprocesar(cfg: PrepConfig) -> Dict:
    """Fase 1."""
    set_seeds(cfg.seed)
    t0 = time.time()
    os.makedirs(cfg.out_dir, exist_ok=True)
    layout = detect_layout(cfg.dataset_root)
    videos = load_videos(cfg.dataset_root, "c23")
    try:
        splits = split_official(videos)
        protocolo = "oficial FF++ (720/140/140)"
    except ValueError:
        splits = split_by_identity(videos, (0.70, 0.15, 0.15), cfg.seed)
        protocolo = "por identidad (videos fuera de la lista oficial)"
    splits = {k: _limit(vs, cfg.max_videos_per_split, cfg.seed) for k, vs in splits.items()}
    leak = check_identity_leakage(splits)
    assert not any(leak.values()), f"Fuga de identidad: {leak}"
    print(f"Estructura: {layout} | protocolo: {protocolo}")
    for k, vs in splits.items():
        print(f"  {k:5s}: {sum(v.label == 0 for v in vs):4d} reales | {sum(v.label == 1 for v in vs):5d} fakes")

    # Caché principal C23: train + val + test (un solo índice de videos)
    allv = splits["train"] + splits["val"] + splits["test"]
    budget = {"val": cfg.budget_val, "test": cfg.budget_test}
    budgets = [(cfg.budget_train_real if v.label == 0 else cfg.budget_train_fake) if v.split == "train"
               else budget[v.split] for v in allv]
    print("Extrayendo rostros C23…")
    c23 = build_face_cache_dir(allv, os.path.join(cfg.out_dir, "cache_c23"), cfg.n_grid, budgets, cfg.workers)
    caches = {"c23": "cache_c23"}

    # C40 real: mismos videos de test en compresión alta (solo con la estructura oficial)
    extra = {}
    if layout == "ffpp":
        try:
            v40 = {v.video_id: v for v in load_videos(cfg.dataset_root, "c40")}
            t40 = [v40[v.video_id] for v in splits["test"] if v.video_id in v40]
            if len(t40) < len(splits["test"]):
                print(f"AVISO: C40 tiene {len(t40)}/{len(splits['test'])} videos de test")
            if t40:
                for v in t40:
                    v.split = "test"
                assign_identity_groups(t40)
                print("Extrayendo rostros C40 (test)…")
                build_face_cache_dir(t40, os.path.join(cfg.out_dir, "cache_c40_test"), cfg.n_grid,
                                     [cfg.budget_test] * len(t40), cfg.workers)
                caches["c40"] = "cache_c40_test"
                extra["c40"] = t40
        except FileNotFoundError:
            print("Sin versión C40 en el dataset: se omite la prueba de mala calidad real")
    if cfg.celebdf_root:
        cdf = list_videos_celebdf(cfg.celebdf_root)
        assign_identity_groups(cdf)
        print(f"Extrayendo rostros Celeb-DF v2 ({len(cdf)} videos de test)…")
        build_face_cache_dir(cdf, os.path.join(cfg.out_dir, "cache_celebdf_test"), cfg.n_grid,
                             [cfg.budget_test] * len(cdf), cfg.workers)
        caches["celebdf"] = "cache_celebdf_test"
        extra["celebdf"] = cdf

    # Normalización robusta: estadísticos solo de frames de train (rostro y ambos modos del mapa)
    rng = np.random.RandomState(cfg.seed)
    tr_videos = np.array([i for i, v in enumerate(allv) if v.split == "train"])
    tr_frames = np.flatnonzero(np.isin(c23["frame_video"], tr_videos))
    fit_idx = np.sort(rng.choice(tr_frames, size=min(cfg.norm_fit_frames, len(tr_frames)), replace=False))
    faces, maps = [], {m: [] for m in DIFF_MODES}
    for s in range(0, len(fit_idx), 64):
        x = bk.asarray(np.asarray(c23["faces"][fit_idx[s:s + 64]])).transpose(0, 3, 1, 2) / 255.0
        faces.append(bk.to_numpy(x))
        for m in DIFF_MODES:
            maps[m].append(bk.to_numpy(bilateral_diff_map(x, m)))
    norm = {"rostro": RobustNormalizer(clip=5.0).fit(np.concatenate(faces)).to_dict(),
            **{f"mapa_{m}": RobustNormalizer(clip=5.0).fit(np.concatenate(maps[m])).to_dict() for m in DIFF_MODES},
            "n_frames_train": int(len(fit_idx))}

    def vrec(v: VideoItem) -> Dict:
        return {"video_id": v.video_id, "label": int(v.label), "method": v.method, "split": v.split,
                "group": int(v.group), "path": v.path}

    manifest = {"protocolo": protocolo, "layout": layout, "caches": caches, "normalizacion": norm,
                "videos_c23": [vrec(v) for v in allv],
                **{f"videos_{k}": [vrec(v) for v in vs] for k, vs in extra.items()},
                "fugas": leak, "config": asdict(cfg), "minutos": (time.time() - t0) / 60}
    json.dump(manifest, open(os.path.join(cfg.out_dir, "manifest.json"), "w"), indent=1, ensure_ascii=False)
    print(f"Fase 1 completa en {manifest['minutos']:.1f} min → {cfg.out_dir}/manifest.json")
    return manifest


# Fase 2
@dataclass
class TrainSetup:
    prep_dir: str
    out_dir: str
    experiment: str = "robusto_con_aumento"
    seed: int = 42
    diff_mode: str = "residual"
    yaw_mask: float = 0.0
    epochs: int = 15
    patience: int = 4
    frames_per_video: int = 8  # frames de cada real por época; los fakes se equilibran
    lr: float = 1e-4
    swin_lr_mult: float = 0.1
    batch_size: int = 16
    eval_batch_size: int = 32
    swin_pretrained: bool = True
    weights_path: Optional[str] = None
    extra: Dict = field(default_factory=dict)


def make_model(branches: str, pretrained: bool, weights_path: Optional[str], lr_mult: float, seed: int):
    from cnn import CNNBranch
    from detector import DeepfakeDetectorScratch
    from mlp import MLPClassifier
    from pretrained import download, load_swin_imagenet, set_lr_mult
    from swin import SwinBranch

    bk.set_seed(seed)
    swin = SwinBranch()
    if pretrained:
        load_swin_imagenet(swin, weights_path or download(), verbose=False)
        for m in [swin.patch_embed, *swin.layers, swin.norm]:
            set_lr_mult(m, lr_mult)
    return DeepfakeDetectorScratch(swin, CNNBranch(3, (32, 64, 128), 256), MLPClassifier(512, 256, 2, 0.3),
                                   drop_branch=0.1, branches=branches)


def entrenar_y_evaluar(st: TrainSetup) -> str:
    """Fase 2."""
    from trainer import Experiment, TrainConfig

    assert st.experiment in EXPERIMENTS, f"experimento desconocido: {st.experiment}"
    set_seeds(st.seed)
    t0 = time.time()
    os.makedirs(st.out_dir, exist_ok=True)
    man = json.load(open(os.path.join(st.prep_dir, "manifest.json")))
    c23 = load_face_cache_dir(os.path.join(st.prep_dir, man["caches"]["c23"]))
    vids = man["videos_c23"]
    has_face = c23["n_detect"] > 0  # videos sin ningún rostro detectado no se pueden puntuar
    split_videos = {k: np.array([i for i, v in enumerate(vids) if v["split"] == k and has_face[i]])
                    for k in ("train", "val", "test")}
    n_sin = int((~has_face).sum())
    if n_sin:
        print(f"AVISO: {n_sin} videos sin rostros detectados quedan fuera (se reporta en meta)")
    face_norm = RobustNormalizer.from_dict(man["normalizacion"]["rostro"])
    diff_norm = RobustNormalizer.from_dict(man["normalizacion"][f"mapa_{st.diff_mode}"])
    branches, use_aug = EXPERIMENTS[st.experiment]
    model = make_model(branches, st.swin_pretrained, st.weights_path, st.swin_lr_mult, st.seed)
    tcfg = TrainConfig(epochs=st.epochs, batch_size=st.batch_size, eval_batch_size=st.eval_batch_size, lr=st.lr,
                       frames_per_video=st.frames_per_video, patience=st.patience, focal_alpha=0.5,
                       seed=st.seed, diff_mode=st.diff_mode, yaw_mask=st.yaw_mask, balance_classes=True)
    name = f"{st.experiment}_s{st.seed}"
    bk.set_seed(st.seed)
    exp = Experiment(name, model, tcfg, c23, split_videos, face_norm, diff_norm,
                     QualityAugmenter(AugConfig()) if use_aug else None, QUALITY_LEVELS, st.out_dir)
    exp.fit()

    rows = []

    def add(cond: str, vlist: List[Dict], idx: np.ndarray, scores: np.ndarray) -> None:
        for i, s in zip(idx, scores):
            v = vlist[int(i)]
            rows.append({"experimento": st.experiment, "semilla": st.seed, "condicion": cond,
                         "video_id": v["video_id"], "metodo": v["method"], "grupo": v["group"],
                         "etiqueta": v["label"], "score": float(s)})

    yv, sv = exp.video_predictions("val")
    add("val_c23", vids, split_videos["val"], sv)
    for cond, spec in SYNTH_TEST.items():
        _, s = exp.video_predictions("test", spec or None)
        add("c23" if cond == "buena" else f"c23_{cond}", vids, split_videos["test"], s)
    for key in ("c40", "celebdf"):
        if key in man["caches"]:
            cache = load_face_cache_dir(os.path.join(st.prep_dir, man["caches"][key]))
            vv, _, s = exp.predict_on(cache)
            add(key, man[f"videos_{key}"], vv, s)
    import pandas as pd

    df = pd.DataFrame(rows)
    path = os.path.join(st.out_dir, f"scores_{name}.csv")
    df.to_csv(path, index=False)
    json.dump({"setup": asdict(st), "historial": exp.history, "mejor_epoca": exp.best_epoch,
               "minutos": (time.time() - t0) / 60, "backend": "gpu" if bk.ON_GPU else "cpu",
               "videos_sin_rostro": n_sin},
              open(os.path.join(st.out_dir, f"meta_{name}.json"), "w"), indent=1)
    from sklearn.metrics import roc_auc_score
    for cond, g in df.groupby("condicion"):
        if g["etiqueta"].nunique() == 2:
            print(f"  {cond:24s} AUC = {roc_auc_score(g['etiqueta'], g['score']):.3f}  ({len(g)} videos)")
    print(f"Fase 2 ({name}) completa en {(time.time() - t0) / 60:.1f} min → {path}")
    return path


# Fase 3
COND_ORDER = ["c23", "c23_regular", "c23_mala", "c23_muy_mala", "c23_jpeg_q30_no_visto", "c40", "celebdf"]
COMPARISONS = [("robusto_con_aumento", "base_sin_aumento", "efecto del aumento de calidad"),
               ("robusto_con_aumento", "ablacion_solo_swin", "aporte de la rama de asimetría"),
               ("robusto_con_aumento", "ablacion_solo_asimetria", "aporte de la rama Swin")]


def analizar(score_globs: Sequence[str], out_dir: str, n_boot: int = 2000) -> Dict:
    """Fase 3."""
    import pandas as pd
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from sklearn.metrics import roc_auc_score, roc_curve

    from metrics import binary_metrics, bootstrap_auc, paired_bootstrap_auc_diff, youden_threshold

    files = sorted({f for g in score_globs for f in glob.glob(g, recursive=True)})
    assert files, f"No hay CSV de puntuaciones en {score_globs}"
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    df = df.drop_duplicates(["experimento", "semilla", "condicion", "video_id"])
    os.makedirs(os.path.join(out_dir, "figuras"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "tablas"), exist_ok=True)
    conds = [c for c in COND_ORDER if c in set(df["condicion"])]
    exps = [e for e in EXPERIMENTS if e in set(df["experimento"])]
    print(f"{len(files)} archivos | experimentos: {exps} | semillas: {sorted(df['semilla'].unique())}")

    # 1) AUC por experimento x condición: por semilla (IC bootstrap) y media ± d.e. entre semillas
    rows = []
    for (e, s, c), g in df[df["condicion"].isin(conds)].groupby(["experimento", "semilla", "condicion"]):
        b = bootstrap_auc(g["etiqueta"].values, g["score"].values, g["grupo"].values, n_boot, seed=42)
        rows.append({"experimento": e, "semilla": s, "condicion": c, **b, "n_videos": len(g)})
    per_seed = pd.DataFrame(rows)
    per_seed.to_csv(os.path.join(out_dir, "tablas", "auc_por_semilla.csv"), index=False)
    summ = per_seed.groupby(["experimento", "condicion"]).agg(
        auc_media=("auc", "mean"), auc_de=("auc", "std"), n_semillas=("auc", "size"),
        ic_inf_medio=("ic_inf", "mean"), ic_sup_medio=("ic_sup", "mean")).reset_index()
    summ.to_csv(os.path.join(out_dir, "tablas", "auc_resumen.csv"), index=False)
    pivot = summ.pivot(index="experimento", columns="condicion", values="auc_media").reindex(index=exps, columns=conds)
    pivot.round(4).to_csv(os.path.join(out_dir, "tablas", "auc_tabla_principal.csv"))
    print("\nAUC medio (entre semillas) por condición:\n", pivot.round(3).to_string())

    # 2) Comparaciones pareadas (mismos videos, misma semilla)
    comp = []
    for a, b_, what in COMPARISONS:
        for c in conds:
            for s in sorted(df["semilla"].unique()):
                ga = df[(df.experimento == a) & (df.semilla == s) & (df.condicion == c)].set_index("video_id")
                gb = df[(df.experimento == b_) & (df.semilla == s) & (df.condicion == c)].set_index("video_id")
                common = ga.index.intersection(gb.index)
                if len(common) == 0 or ga.loc[common, "etiqueta"].nunique() < 2:
                    continue
                r = paired_bootstrap_auc_diff(ga.loc[common, "etiqueta"].values, ga.loc[common, "score"].values,
                                              gb.loc[common, "score"].values, ga.loc[common, "grupo"].values,
                                              n_boot, seed=42)
                comp.append({"comparacion": f"{a} − {b_}", "pregunta": what, "condicion": c, "semilla": s, **r})
    comp = pd.DataFrame(comp, columns=["comparacion", "pregunta", "condicion", "semilla", "diferencia",
                                       "ic_inf", "ic_sup", "p_valor"])
    comp.to_csv(os.path.join(out_dir, "tablas", "comparaciones_pareadas.csv"), index=False)

    # 3) Métricas con umbral (fijado en validación C23 de cada modelo): C23 y C40
    thr_rows = []
    for (e, s), g in df.groupby(["experimento", "semilla"]):
        v = g[g.condicion == "val_c23"]
        if v["etiqueta"].nunique() < 2:
            continue
        thr = youden_threshold(v["etiqueta"].values, v["score"].values)
        for c in [c for c in ("c23", "c40", "celebdf") if c in conds]:
            t = g[g.condicion == c]
            m = binary_metrics(t["etiqueta"].values, t["score"].values, thr)
            thr_rows.append({"experimento": e, "semilla": s, "condicion": c, **m})
    pd.DataFrame(thr_rows if thr_rows else [{"experimento": None}]).dropna(how="all").to_csv(os.path.join(out_dir, "tablas", "metricas_con_umbral.csv"), index=False)

    # 4) Figuras (300 dpi)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    x = np.arange(len(conds))
    for k, e in enumerate(exps):
        sub = summ[summ.experimento == e].set_index("condicion").reindex(conds)
        ax.errorbar(x + (k - len(exps) / 2) * 0.06, sub["auc_media"], yerr=sub["auc_de"].fillna(0),
                    marker="o", capsize=3, lw=1.6, label=e)
    ax.set_xticks(x, conds, rotation=25, ha="right")
    ax.axhline(0.5, ls="--", c="gray", lw=1)
    ax.set_ylabel("AUC (media ± d.e. entre semillas)")
    ax.set_title("AUC por condición de test")
    ax.legend(fontsize=8)
    fig.savefig(os.path.join(out_dir, "figuras", "fig_ffpp_01_auc_por_condicion.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    seed0 = int(sorted(df["semilla"].unique())[0])
    roc_conds = [c for c in ("c23", "c40", "celebdf") if c in conds]
    fig, axs = plt.subplots(1, len(roc_conds), figsize=(4.6 * len(roc_conds), 4.2), squeeze=False)
    for a_, c in zip(axs[0], roc_conds):
        for e in exps:
            g = df[(df.experimento == e) & (df.semilla == seed0) & (df.condicion == c)]
            if g["etiqueta"].nunique() == 2:
                f, t, _ = roc_curve(g["etiqueta"], g["score"])
                a_.plot(f, t, lw=1.5, label=f"{e} ({roc_auc_score(g['etiqueta'], g['score']):.3f})")
        a_.plot([0, 1], [0, 1], "--", c="gray", lw=1)
        a_.set_title(f"ROC: {c}: semilla {seed0}")
        a_.set_xlabel("FPR")
        a_.set_ylabel("TPR")
        a_.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "figuras", "fig_ffpp_02_roc.png"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    if len(comp):
        cm = comp.groupby(["comparacion", "condicion"]).agg(dif=("diferencia", "mean"), lo=("ic_inf", "mean"),
                                                            hi=("ic_sup", "mean")).reset_index()
        fig, ax = plt.subplots(figsize=(8, 4.2))
        comps = list(dict.fromkeys(cm["comparacion"]))
        for k, cname in enumerate(comps):
            sub = cm[cm.comparacion == cname].set_index("condicion").reindex(conds).dropna()
            xs = np.array([conds.index(c) for c in sub.index]) + (k - len(comps) / 2) * 0.12
            ax.errorbar(xs, sub["dif"], yerr=[sub["dif"] - sub["lo"], sub["hi"] - sub["dif"]], fmt="o",
                        capsize=3, label=cname)
        ax.axhline(0, c="gray", lw=1)
        ax.set_xticks(np.arange(len(conds)), conds, rotation=25, ha="right")
        ax.set_ylabel("Δ AUC (IC 95 % bootstrap)")
        ax.set_title("Comparaciones pareadas")
        ax.legend(fontsize=7)
        fig.savefig(os.path.join(out_dir, "figuras", "fig_ffpp_03_comparaciones.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)
    res = {"archivos": files, "condiciones": conds, "experimentos": exps,
           "auc_tabla_principal": pivot.round(4).to_dict()}
    json.dump(res, open(os.path.join(out_dir, "resumen_analisis.json"), "w"), indent=1, ensure_ascii=False)
    return res


In [ ]:
sys.path.insert(0, os.path.abspath("code_kaggle"))
import backend as bk
bk.set_dtype(np.float32); bk.set_seed(SEED)
print("Backend:", "CuPy (GPU)" if bk.ON_GPU else "NumPy (CPU)")

In [ ]:
from data import find_dataset_dir
from experimento import PrepConfig, preprocesar
ff_root = find_dataset_dir([INPUT_ROOT])
assert ff_root, "No se encontró FF++ (original_sequences/ + manipulated_sequences/, o real/ + fake/) en el input"
cdf = [os.path.dirname(p) for p in glob.glob(os.path.join(INPUT_ROOT, "**", "List_of_testing_videos.txt"), recursive=True)]
print("FF++:", ff_root, "| Celeb-DF:", cdf[0] if cdf else "no añadido")
cfg = PrepConfig(dataset_root=ff_root, out_dir="prep", celebdf_root=cdf[0] if cdf else None,
                 workers=max(1, min(4, os.cpu_count() or 1)),
                 max_videos_per_split=10 if SMOKE else None)
manifest = preprocesar(cfg)

In [ ]:
import collections
for k in [k for k in manifest if k.startswith("videos_")]:
    c = collections.Counter((v["split"], "real" if v["label"] == 0 else "fake") for v in manifest[k])
    print(k, dict(sorted(c.items())))
print("Fugas de identidad:", manifest["fugas"])
for c in manifest["caches"].values():
    m = np.load(os.path.join("prep", c, "meta.npz"))
    print(f"{c}: {len(m['frame_video'])} rostros | videos sin rostro: {int((m['n_detect'] == 0).sum())}")